# 🌿 Deteksi & Klasifikasi Penyakit Daun Teh Terintegrasi (Audit-Compliant Edition v3):
## **YOLO26n + Coordinate Attention (Deteksi Bercak) & YOLO26n-seg (Segmentasi Daun Individual) + Leaf–Lesion Association**

Notebook ini telah direvisi secara menyeluruh sesuai **Audit Metodologi & Kode Tahap Lanjut (Poin A–H)**:

### 1. Ringkasan Perbaikan Audit Teknis & Metodologi (Poin A – H)
| Kode Audit | Cell Terkait | Temuan Audit | Perbaikan yang Diterapkan |
| :---: | :---: | :--- | :--- |
| **A** | **Cell 2, 14, 16, 22, 34** | Sebelumnya mengecualikan `cid == 3` (`Helopeltis`, kelas penyakit utama) dan belum menyaring `cid == 6` (`Healty Leaf`). | Mendefinisikan `HEALTHY_LEAF_CLASS_ID = 6` dan `DISEASE_LESION_CLASS_IDS = (0, 1, 2, 3, 4, 5)`. **Kelas 3 (`Helopeltis`) disertakan penuh sebagai bercak penyakit**, sedangkan **Kelas 6 (`Healty Leaf`) disaring dari deteksi/pseudo-label/evaluasi bercak**. |
| **B** | **Cell 2, 24, 28** | Cell 24 & 28 memanggil `compute_md5(img_p)` sementara Cell 2 mendefinisikan `compute_file_md5(filepath)`. | Mengganti seluruh pemanggilan di Cell 24 & 28 menjadi `compute_file_md5(img_p)` (serta menyediakan alias kompatibilitas di Cell 2). |
| **C** | **Cell 2, 24, 28, 30** | Penggunaan `yaml.dump(...)` tanpa `import yaml`. | Menambahkan `pyyaml` pada instalasi dependensi dan `import yaml` pada **Cell 2, 24, 28, dan 30**. |
| **D** | **Cell 16, 18, 22, 28, 30** | Cell 16 memuat `yolo26n-seg.pt` umum sebelum model daun dilatih di Cell 30; Cell 22 memanggil `analyze_field_image()` sehingga bercak hilang bila segmentasi daun gagal. | **Cell 16 tidak lagi memuat `yolo26n-seg.pt` COCO umum** sebagai pengganti model daun teh. **Cell 22 menggunakan `detect_lesions()` secara independen** tanpa bergantung pada segmentasi daun. **Cell 28 & 30 wajib dijalankan terlebih dahulu** sebelum pengujian terintegrasi di Cell 18, 32, dan 34. |
| **E** | **Cell 22 & 24** | Fungsi verifikasi di Cell 22 dapat menyetujui file hanya karena formatnya valid, dan gambar tanpa prediksi bercak tidak ikut diperiksa. | Menambahkan **persetujuan eksplisit anotator (`annotator_id` & `review_decision` = `APPROVED_POSITIVE` / `APPROVED_NEGATIVE_NO_LESION`)** serta memasukkan **gambar tanpa prediksi bercak (`ZERO_PRED_MUST_CHECK_MISSED_LESIONS`)** ke dalam alur review manual untuk menangkap bercak kecil yang terlewat. |
| **F** | **Cell 2, 24, 28** | Pembentukan path label Roboflow menghasilkan `train/labels/images/nama.txt` dan menyalin ulang gambar dengan nama Roboflow sehingga menduplikasi foto. | Menambahkan fungsi `find_matching_yolo_label_file()` (mendukung struktur standar Roboflow `train/labels/nama.txt`) dan `match_image_to_field_manifest()` yang memetakan anotasi ke nama file kanonik asli (`dst_stem`) **tanpa menduplikasi foto**. |
| **G** | **Cell 28 & 30** | `val_verified=True` hanya memeriksa jumlah gambar `> 0`, belum menjamin seluruh daun target dianotasi atau kasus sulit tidak dilewati. | Mewajibkan **catatan verifikasi manual per gambar (`leaf_seg_manual_verification_checklist.csv`)** dengan `all_target_leaves_annotated=True`, `manual_verified=True`, dan **cakupan 100% citra split `val`/`test` asli (`missing_images == 0`)** tanpa melewati kasus daun bertumpuk/sulit. |
| **H** | **Cell 34** | Ground truth pasangan daun–bercak ditentukan otomatis dari overlap terbesar, yang rawan salah pada daun bertumpuk. | Menambahkan **manifest Ground Truth pasangan daun–bercak terverifikasi manual (`gt_leaf_lesion_pairs_test_verified.csv`)** dengan `verified_gt_leaf_id` dan `manual_verified=True` untuk menghitung Pairwise Precision, Recall, dan F1 secara sah. |

### 2. Urutan Eksekusi Pipeline yang Direkomendasikan
1. **Tahap Dasar & Detektor Bercak**: Jalankan **Cell 2 → Cell 4 → Cell 6 → Cell 8 → Cell 10 → Cell 12 → Cell 14**.
2. **Tahap Adaptasi Lapangan (Independen dari Segmentasi Daun)**: Jalankan **Cell 16 (inisialisasi detektor bercak) → Cell 22 (`detect_lesions` independen + verifikasi manual eksplisit) → Cell 24 (Fine-Tuning) → Cell 26 (Evaluasi Fine-Tuning)**.
3. **Tahap Segmentasi Daun Individual**: Jalankan **Cell 28 (Dataset Segmentasi Terverifikasi) → Cell 30 (Validasi Poligon & Training `YOLO26n-seg` Khusus Daun Teh)**.
4. **Tahap Integrasi & Evaluasi Pasangan Daun–Bercak**: Setelah Cell 30 selesai melatih `LEAF_SEG_WEIGHTS`, jalankan **Cell 18 → Cell 32 → Cell 34 → Cell 20 (Backup)**.

---

### [Cell 2 | Prioritas 2] Persiapan Dataset: Pemisahan Single Leaf vs 500 Foto Lapangan dengan Pembagian Berbasis Tanaman / Sesi Foto (*Group-Aware Split*)

In [ ]:
import os
import re
import sys
import glob
import json
import hashlib
import random
import shutil
import zipfile
from pathlib import Path
from collections import defaultdict, Counter

# 1. Mount Google Drive (jika dijalankan di Google Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IS_COLAB = True
except ImportError:
    IS_COLAB = False
    print("[INFO] Berjalan di lingkungan lokal (non-Colab).")

if IS_COLAB:
    !pip install -q ultralytics opencv-python-headless matplotlib pandas scikit-learn pyyaml

import yaml
import cv2
import numpy as np
import pandas as pd
import torch

print(f"[INFO] PyTorch: {torch.__version__} | CUDA Tersedia: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"[INFO] GPU: {torch.cuda.get_device_name(0)}")

BASE_WORK_DIR = "/content" if IS_COLAB else os.path.abspath("./workspace_revised")
os.makedirs(BASE_WORK_DIR, exist_ok=True)

SINGLE_LEAF_DIR  = os.path.join(BASE_WORK_DIR, "tea_yolo_single_leaf")
SINGLE_LEAF_YAML = os.path.join(SINGLE_LEAF_DIR, "data_single_leaf.yaml")
DATA_DIR  = SINGLE_LEAF_DIR
DATA_YAML = SINGLE_LEAF_YAML

FIELD_500_DIR       = os.path.join(BASE_WORK_DIR, "tea_field_500")
FIELD_500_YAML      = os.path.join(FIELD_500_DIR, "data_field_500.yaml")
FIELD_MANIFEST_JSON = os.path.join(FIELD_500_DIR, "field_500_split_manifest.json")

CLASS_NAMES = [
    "Tea algal leaf spot",  # 0 (Penyakit bercak)
    "Brown Blight",         # 1 (Penyakit bercak)
    "Gray Blight",          # 2 (Penyakit bercak)
    "Helopeltis",           # 3 (Penyakit/hama bercak tusukan - WAJIB DISERTAKAN!)
    "Red Spider",           # 4 (Penyakit/hama bercak)
    "Green Mirid Bug",      # 5 (Penyakit/hama bercak)
    "Healty Leaf",          # 6 (Healthy Leaf - anotasi level daun utuh, disaring dari deteksi bercak)
]
HEALTHY_LEAF_CLASS_ID    = 6
DISEASE_LESION_CLASS_IDS = (0, 1, 2, 3, 4, 5)

def compute_file_md5(filepath: str) -> str:
    h = hashlib.md5()
    with open(filepath, "rb") as f:
        for chunk in iter(lambda: f.read(65536), b""):
            h.update(chunk)
    return h.hexdigest()

# Alias kompatibilitas agar pemanggilan compute_file_md5 / compute_md5 selalu konsisten (Revisi Poin B)
compute_md5 = compute_file_md5

def compute_perceptual_dhash(filepath: str, hash_size: int = 8) -> int:
    """Menghitung 64-bit perceptual difference hash (dHash) untuk mendeteksi foto beruntun dari daun/tanaman yang sama."""
    img = cv2.imread(filepath, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return 0
    resized = cv2.resize(img, (hash_size + 1, hash_size), interpolation=cv2.INTER_AREA)
    diff = resized[:, 1:] > resized[:, :-1]
    bit_val = 0
    for b in diff.flatten():
        bit_val = (bit_val << 1) | int(b)
    return bit_val

def hamming_distance_64(h1: int, h2: int) -> int:
    return bin(h1 ^ h2).count("1")

def canonical_image_stem(filename: str) -> str:
    stem = os.path.splitext(os.path.basename(filename))[0].lower()
    stem = re.sub(r"^(bal_\d+_\d+_|aug_\d+_|copy_|vfield_|field_(train|val|test)_\d+_)", "", stem)
    stem = re.sub(r"\.rf\.[a-f0-9]+$", "", stem)
    return stem

def extract_plant_or_session_group_id(filepath: str) -> str:
    """
    REVISI CELL 2 (Pembagian Berdasarkan Tanaman atau Sesi Foto):
    Mengekstrak identitas grup tanaman / pohon / sesi pemotretan dari:
      1. Nama subfolder tanaman/plot/sesi (jika diorganisir per folder tanaman/sesi).
      2. Pola ID tanaman eksplisit pada nama file (mis. 'tanaman05_...', 'plant_12_...', 'pohon3_...', 'bush08_...').
      3. Pola timestamp sesi kamera (mis. 'IMG_20261012_1430xx' -> grup sesi 'session_20261012_143' per blok 10 menit).
      4. Pola nomor urut burst kamera (mis. 'DSC_0142' -> dikelompokkan per blok 5 frame berurutan agar burst objek yang sama menyatu).
    """
    parent_name = os.path.basename(os.path.dirname(filepath)).lower()
    if parent_name not in ("train", "val", "valid", "test", "images", "data_lapangan", "dataset_lapangan_500", ""):
        return f"folder_{parent_name}"

    c_stem = canonical_image_stem(filepath)
    # Cek pola eksplisit ID tanaman / blok / sesi
    m_plant = re.search(r"(tanaman|plant|pohon|bush|tree|plot|sesi|session)[_\-]?(\d+)", c_stem)
    if m_plant:
        return f"{m_plant.group(1)}_{int(m_plant.group(2)):04d}"

    # Cek pola timestamp kamera: IMG_YYYYMMDD_HHMMSS atau YYYY-MM-DD_HH-MM
    m_ts = re.search(r"(20\d{6})[_\-](\d{4})\d{2}", c_stem)
    if m_ts:
        date_part, hhmm = m_ts.group(1), m_ts.group(2)
        # Kelompokkan per jendela sesi 10 menit (HHM_)
        return f"session_{date_part}_{hhmm[:3]}0"

    # Cek awalan seri + nomor urut kamera (burst grouping per 5 nomor berurutan)
    m_seq = re.match(r"^([a-z]+)[_\-]?(\d{3,6})", c_stem)
    if m_seq:
        prefix, num_str = m_seq.group(1), m_seq.group(2)
        burst_bucket = int(num_str) // 5
        return f"burst_{prefix}_{burst_bucket:05d}"

    # Fallback ke akar kata utama sebelum suffix indeks (_1, _2, _a, _b, dll.)
    base_root = re.sub(r"[_\-]?(crop|view|shot|angle|part)?[_\-]?\d+$", "", c_stem)
    return f"stem_{base_root or c_stem}"

# 3. Ekstrak Dataset Single Leaf (tea_yolo.zip) ke SINGLE_LEAF_DIR
if not os.path.exists(os.path.join(SINGLE_LEAF_DIR, "images", "train")):
    existing_dirs = ["/content/tea_yolo", "D:/SKRIPSI/tea_yolo"]
    copied_local = False
    for ed in existing_dirs:
        if os.path.exists(os.path.join(ed, "images", "train")):
            print(f"[INFO] Menyalin dataset Single Leaf dari {ed} ke {SINGLE_LEAF_DIR} ...")
            shutil.copytree(ed, SINGLE_LEAF_DIR, dirs_exist_ok=True)
            copied_local = True
            break
    if not copied_local:
        zip_candidates = (
            glob.glob('/content/drive/MyDrive/SKRIPSI/tea_yolo.zip') +
            glob.glob('/content/drive/MyDrive/**/tea_yolo.zip', recursive=True) +
            glob.glob('D:/SKRIPSI/tea_yolo.zip')
        )
        if not zip_candidates:
            raise FileNotFoundError("Berkas 'tea_yolo.zip' tidak ditemukan di Google Drive maupun lokal!")
        zip_path = zip_candidates[0]
        tmp_extract = os.path.join(BASE_WORK_DIR, "_tmp_tea_yolo_extract")
        os.makedirs(tmp_extract, exist_ok=True)
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall(tmp_extract)
        extracted_sub = os.path.join(tmp_extract, "tea_yolo")
        src_move = extracted_sub if os.path.exists(extracted_sub) else tmp_extract
        shutil.copytree(src_move, SINGLE_LEAF_DIR, dirs_exist_ok=True)
        shutil.rmtree(tmp_extract, ignore_errors=True)

def sanitize_and_verify_single_leaf_splits(single_leaf_dir: str):
    """
    Membersihkan kontaminasi file lapangan dari dataset Single Leaf dan memastikan
    tidak ada MD5 hash, canonical stem, maupun grup sesi tanaman yang bocor antar split.
    """
    removed_contamination = 0
    removed_leaks = 0
    for sp in ("train", "val", "test"):
        for kind in ("images", "labels"):
            folder = os.path.join(single_leaf_dir, kind, sp)
            if not os.path.exists(folder):
                continue
            for fp in glob.glob(os.path.join(folder, "*.*")):
                bname = os.path.basename(fp)
                if bname.startswith(("lapangan_", "rf_", "field_", "vfield_", "bal_")):
                    os.remove(fp)
                    removed_contamination += 1

    seen_hashes = {"test": set(), "val": set(), "train": set()}
    seen_stems  = {"test": set(), "val": set(), "train": set()}
    split_counts = {}

    for sp in ("test", "val", "train"):
        img_folder = os.path.join(single_leaf_dir, "images", sp)
        lbl_folder = os.path.join(single_leaf_dir, "labels", sp)
        if not os.path.exists(img_folder):
            split_counts[sp] = 0
            continue
        imgs = sorted([
            p for p in glob.glob(os.path.join(img_folder, "*.*"))
            if p.lower().endswith((".jpg", ".jpeg", ".png"))
        ])
        kept_count = 0
        for img_p in imgs:
            f_hash = compute_file_md5(img_p)
            c_stem = canonical_image_stem(img_p)
            prior_splits = ["test"] if sp == "val" else (["test", "val"] if sp == "train" else [])
            is_leaked = any(
                (f_hash in seen_hashes[psp]) or (c_stem in seen_stems[psp])
                for psp in prior_splits
            )
            if is_leaked:
                raw_stem = os.path.splitext(os.path.basename(img_p))[0]
                lbl_p = os.path.join(lbl_folder, raw_stem + ".txt")
                os.remove(img_p)
                if os.path.exists(lbl_p):
                    os.remove(lbl_p)
                removed_leaks += 1
            else:
                seen_hashes[sp].add(f_hash)
                seen_stems[sp].add(c_stem)
                kept_count += 1
        split_counts[sp] = kept_count

    for cache_f in glob.glob(os.path.join(single_leaf_dir, "labels", "*.cache")):
        os.remove(cache_f)

    with open(SINGLE_LEAF_YAML, "w", encoding="utf-8") as f:
        f.write(
            f"path: {single_leaf_dir}\n"
            f"train: images/train\n"
            f"val: images/val\n"
            f"test: images/test\n"
            f"nc: {len(CLASS_NAMES)}\n"
            f"names: {CLASS_NAMES}\n"
        )

    print("=" * 84)
    print("  [AUDIT DATASET SINGLE LEAF] PEMISAHAN & VERIFIKASI ANTI-BOCOR SELESAI")
    print("=" * 84)
    print(f"  - Split Train : {split_counts.get('train', 0):4d} citra | Val : {split_counts.get('val', 0):4d} citra | Test : {split_counts.get('test', 0):4d} citra")
    print(f"  - Kontaminasi dibersihkan: {removed_contamination} berkas | Duplikat lintas-split dihapus: {removed_leaks} berkas")
    return seen_hashes

single_leaf_hashes = sanitize_and_verify_single_leaf_splits(SINGLE_LEAF_DIR)

# ==============================================================================
# B. PEMBAGIAN 500 GAMBAR LAPANGAN BERDASARKAN TANAMAN / SESI FOTO (GROUP SPLIT)
# ==============================================================================
def prepare_field_500_group_aware_dataset(
    field_dst_dir: str,
    single_leaf_hash_dict: dict,
    max_field_images: int = 500,
    split_ratios: tuple = (0.70, 0.15, 0.15),
    dhash_hamming_thr: int = 6,
    seed: int = 42,
):
    """
    REVISI CELL 2 (Prioritas 2):
    Mengganti pembagian acak per gambar dengan PEMBAGIAN BERBASIS GRUP TANAMAN / SESI FOTO
    (Group-Aware Split) yang dikombinasikan dengan klasterisasi kemiripan visual (dHash):
      1. Setiap foto diekstrak ID grup tanaman/sesi/burst-nya (extract_plant_or_session_group_id).
      2. Foto yang memiliki kemiripan visual tinggi (Hamming distance dHash <= 6, misalnya
         dua sudut pengambilan dari helai daun/pohon yang sama) digabungkan ke dalam klaster grup yang sama.
      3. Seluruh anggota dalam satu grup tanaman/sesi dialokasikan BERSAMA-SAMA ke dalam satu split
         (train ATAU val ATAU test), sehingga 0% gambar serupa tersebar antar split!
    """
    for sp in ("train", "val", "test"):
        os.makedirs(os.path.join(field_dst_dir, "images", sp), exist_ok=True)
        os.makedirs(os.path.join(field_dst_dir, "labels", sp), exist_ok=True)

    field_src_candidates = [
        "/content/drive/MyDrive/SKRIPSI/data_lapangan",
        "/content/drive/MyDrive/SKRIPSI/dataset_lapangan_500",
        "D:/SKRIPSI/data_lapangan",
    ]
    raw_field_imgs = []
    for cand_dir in field_src_candidates:
        if os.path.exists(cand_dir):
            raw_field_imgs = sorted([
                p for p in glob.glob(os.path.join(cand_dir, "**", "*.*"), recursive=True)
                if p.lower().endswith((".jpg", ".jpeg", ".png"))
            ])
            if raw_field_imgs:
                print(f"[INFO] Ditemukan {len(raw_field_imgs)} foto lapangan dari: {cand_dir}")
                break

    all_sl_hashes = set().union(*single_leaf_hash_dict.values())
    unique_field_items = []
    seen_field_hashes = set()

    for img_p in raw_field_imgs:
        f_hash = compute_file_md5(img_p)
        if f_hash in all_sl_hashes or f_hash in seen_field_hashes:
            continue
        seen_field_hashes.add(f_hash)
        stem = os.path.splitext(os.path.basename(img_p))[0]
        lbl_cand = os.path.splitext(img_p)[0] + ".txt"
        init_group = extract_plant_or_session_group_id(img_p)
        dhash_val = compute_perceptual_dhash(img_p)

        # Jika secara visual sangat mirip (dHash Hamming <= dhash_hamming_thr) dengan item sebelumnya, satukan grupnya!
        assigned_group = init_group
        for prev_item in unique_field_items:
            if hamming_distance_64(dhash_val, prev_item["dhash"]) <= dhash_hamming_thr:
                assigned_group = prev_item["group_id"]
                break

        unique_field_items.append({
            "src_img": img_p,
            "src_lbl": lbl_cand if os.path.exists(lbl_cand) else None,
            "md5": f_hash,
            "dhash": dhash_val,
            "stem": stem,
            "canonical_stem": canonical_image_stem(img_p),
            "group_id": assigned_group,
        })
        if len(unique_field_items) >= max_field_images:
            break

    # Kelompokkan item berdasarkan group_id (tanaman / sesi foto)
    groups_dict = {}
    for item in unique_field_items:
        groups_dict.setdefault(item["group_id"], []).append(item)

    group_keys = sorted(groups_dict.keys())
    rng = random.Random(seed)
    rng.shuffle(group_keys)

    n_total = len(unique_field_items)
    target_tr = int(round(n_total * split_ratios[0]))
    target_vl = int(round(n_total * split_ratios[1]))

    splits_map = {"train": [], "val": [], "test": []}
    split_groups = {"train": set(), "val": set(), "test": set()}

    # Alokasikan seluruh isi grup ke satu split yang sama (Greedy Group Packing)
    for g_id in group_keys:
        g_items = groups_dict[g_id]
        if len(splits_map["train"]) < target_tr:
            target_sp = "train"
        elif len(splits_map["val"]) < target_vl:
            target_sp = "val"
        else:
            target_sp = "test"
        splits_map[target_sp].extend(g_items)
        split_groups[target_sp].add(g_id)

    # Verifikasi matematis: 0% irisan grup tanaman/sesi antar split!
    assert len(split_groups["train"] & split_groups["val"]) == 0, "Kebocoran grup tanaman train-val!"
    assert len(split_groups["train"] & split_groups["test"]) == 0, "Kebocoran grup tanaman train-test!"
    assert len(split_groups["val"] & split_groups["test"]) == 0, "Kebocoran grup tanaman val-test!"

    manifest_records = {
        "seed": seed,
        "split_method": "plant_and_session_group_aware",
        "total_images": n_total,
        "total_groups": len(group_keys),
        "group_counts_per_split": {k: len(v) for k, v in split_groups.items()},
        "splits": {"train": [], "val": [], "test": []},
        "stem_to_split": {},
        "md5_to_split": {},
        "stem_to_entry": {},
        "md5_to_entry": {},
    }

    for sp, items in splits_map.items():
        for idx, item in enumerate(items, 1):
            ext = os.path.splitext(item["src_img"])[1].lower()
            dst_name = f"field_{sp}_{idx:03d}_{item['stem']}{ext}"
            dst_stem = os.path.splitext(dst_name)[0]
            dst_img = os.path.join(field_dst_dir, "images", sp, dst_name)
            if not os.path.exists(dst_img):
                shutil.copy2(item["src_img"], dst_img)
            if item["src_lbl"] and os.path.exists(item["src_lbl"]):
                dst_lbl = os.path.join(field_dst_dir, "labels", sp, dst_stem + ".txt")
                shutil.copy2(item["src_lbl"], dst_lbl)
            rec = {
                "split": sp,
                "filename": dst_name,
                "dst_stem": dst_stem,
                "orig_stem": item["stem"],
                "canonical_stem": item["canonical_stem"],
                "group_id": item["group_id"],
                "md5": item["md5"],
                "has_gt_label": bool(item["src_lbl"]),
            }
            manifest_records["splits"][sp].append(rec)
            manifest_records["stem_to_split"][item["canonical_stem"]] = sp
            manifest_records["stem_to_split"][item["stem"].lower()] = sp
            manifest_records["stem_to_split"][dst_stem.lower()] = sp
            manifest_records["md5_to_split"][item["md5"]] = sp
            entry_ref = {"split": sp, "filename": dst_name, "dst_stem": dst_stem, "orig_stem": item["stem"], "md5": item["md5"]}
            manifest_records["md5_to_entry"][item["md5"]] = entry_ref
            manifest_records["stem_to_entry"][item["canonical_stem"]] = entry_ref
            manifest_records["stem_to_entry"][item["stem"].lower()] = entry_ref
            manifest_records["stem_to_entry"][dst_stem.lower()] = entry_ref

    with open(FIELD_MANIFEST_JSON, "w", encoding="utf-8") as f:
        json.dump(manifest_records, f, indent=2)

    with open(FIELD_500_YAML, "w", encoding="utf-8") as f:
        yaml.dump({
            "path": field_dst_dir,
            "train": "images/train",
            "val": "images/val",
            "test": "images/test",
            "nc": len(CLASS_NAMES),
            "names": CLASS_NAMES,
        }, f, sort_keys=False)

    print("\n" + "=" * 84)
    print("  [DATASET LAPANGAN 500 FOTO] PEMBAGIAN BERBASIS TANAMAN / SESI FOTO (GROUP-AWARE)")
    print("=" * 84)
    print(f"  - Direktori Field 500   : {field_dst_dir}")
    print(f"  - Total Foto & Grup     : {n_total} foto dari {len(group_keys)} grup tanaman/sesi unik")
    print(f"  - Field Train (70%)     : {len(splits_map['train']):3d} foto ({len(split_groups['train']):3d} grup tanaman/sesi)")
    print(f"  - Field Val   (15%)     : {len(splits_map['val']):3d} foto ({len(split_groups['val']):3d} grup tanaman/sesi)")
    print(f"  - Field Test  (15%)     : {len(splits_map['test']):3d} foto ({len(split_groups['test']):3d} grup tanaman/sesi)")
    print(f"  - Irisan Grup Lintas Split (Train∩Val, Train∩Test, Val∩Test): 0, 0, 0 (0% Leakage)")

def find_matching_yolo_label_file(img_p: str) -> str:
    """
    REVISI POIN F (Cell 24 & Cell 28):
    Mencari berkas label .txt pasangan dari suatu citra pada berbagai struktur ekspor:
      1. Struktur standar Roboflow YOLO : <root>/<split>/images/<stem>.jpg -> <root>/<split>/labels/<stem>.txt
      2. Struktur standar Dataset YOLO  : <root>/images/<split>/<stem>.jpg -> <root>/labels/<split>/<stem>.txt
      3. Struktur folder datar / berdampingan: <dir>/<stem>.jpg -> <dir>/<stem>.txt atau <dir>/../labels/<stem>.txt
    """
    stem = os.path.splitext(os.path.basename(img_p))[0]
    parent_dir = os.path.dirname(img_p)
    parent_name = os.path.basename(parent_dir).lower()
    grandparent_dir = os.path.dirname(parent_dir)
    grandparent_name = os.path.basename(grandparent_dir).lower()

    candidates = []
    # 1. Struktur Roboflow: train/images/nama.jpg -> train/labels/nama.txt
    if parent_name == "images":
        candidates.append(os.path.join(grandparent_dir, "labels", stem + ".txt"))
    # 2. Struktur Dataset YOLO: images/train/nama.jpg -> labels/train/nama.txt
    if grandparent_name == "images":
        candidates.append(os.path.join(os.path.dirname(grandparent_dir), "labels", os.path.basename(parent_dir), stem + ".txt"))
    # 3. Struktur berdampingan atau subfolder labels:
    candidates.append(os.path.join(parent_dir, stem + ".txt"))
    candidates.append(os.path.join(grandparent_dir, "labels", stem + ".txt"))

    for cand in candidates:
        if os.path.exists(cand):
            return cand
    return None

def match_image_to_field_manifest(img_p: str, field_manifest: dict) -> dict:
    """
    REVISI POIN F (Cell 24 & Cell 28):
    Memetakan citra dari ekspor Roboflow / folder eksternal ke entri kanonik asli
    di `field_500_split_manifest.json` (`{"split": ..., "filename": ..., "dst_stem": ...}`)
    sehingga anotasi disimpan menggunakan nama file kanonik yang sudah ada dan
    TIDAK PERNAH membuat duplikat foto dengan nama `.rf.<hash>.jpg`.
    """
    md5_to_entry = field_manifest.get("md5_to_entry", {})
    stem_to_entry = field_manifest.get("stem_to_entry", {})

    img_md5 = compute_file_md5(img_p)
    if img_md5 in md5_to_entry:
        return md5_to_entry[img_md5]

    raw_stem = os.path.splitext(os.path.basename(img_p))[0].lower()
    clean_rf_stem = raw_stem.split(".rf.")[0]
    canon_stem = canonical_image_stem(img_p)

    for key_cand in (raw_stem, clean_rf_stem, canon_stem):
        if key_cand in stem_to_entry:
            return stem_to_entry[key_cand]

    # Fallback jika manifest lama hanya memiliki stem_to_split:
    stem_to_split = field_manifest.get("stem_to_split", {})
    md5_to_split = field_manifest.get("md5_to_split", {})
    sp = md5_to_split.get(img_md5) or stem_to_split.get(raw_stem) or stem_to_split.get(clean_rf_stem) or stem_to_split.get(canon_stem)
    if sp in ("train", "val", "test"):
        for rec in field_manifest.get("splits", {}).get(sp, []):
            if rec.get("md5") == img_md5 or rec.get("canonical_stem") == canon_stem or rec.get("orig_stem", "").lower() == clean_rf_stem:
                fn = rec["filename"]
                return {"split": sp, "filename": fn, "dst_stem": os.path.splitext(fn)[0], "orig_stem": rec.get("orig_stem"), "md5": rec.get("md5")}
    return None

prepare_field_500_group_aware_dataset(FIELD_500_DIR, single_leaf_hashes, max_field_images=500, seed=42)

# C. Siapkan Bobot Pretrained Dasar: yolo26n.pt & yolo26n-seg.pt
BASE_DET_PT = os.path.join(BASE_WORK_DIR, "yolo26n.pt")
BASE_SEG_PT = os.path.join(BASE_WORK_DIR, "yolo26n-seg.pt")

for pt_name, target_pt in [("yolo26n.pt", BASE_DET_PT), ("yolo26n-seg.pt", BASE_SEG_PT)]:
    if not os.path.exists(target_pt):
        pt_cands = (
            glob.glob(f"/content/drive/MyDrive/SKRIPSI/**/{pt_name}", recursive=True) +
            glob.glob(f"/content/drive/MyDrive/**/{pt_name}", recursive=True) +
            glob.glob(f"D:/SKRIPSI/{pt_name}")
        )
        if pt_cands:
            shutil.copy2(pt_cands[0], target_pt)
            print(f"[OK] {pt_name} disalin dari: {pt_cands[0]}")
        else:
            try:
                from ultralytics import YOLO
                _ = YOLO(pt_name)
                if os.path.exists(pt_name) and os.path.abspath(pt_name) != os.path.abspath(target_pt):
                    shutil.copy2(pt_name, target_pt)
                print(f"[OK] {pt_name} siap melalui Ultralytics.")
            except Exception as e:
                print(f"[INFO] {pt_name} akan diunduh otomatis saat inisialisasi model ({e}).")


### [Cell 4 | Prioritas 3] Coordinate Attention (Tanpa Re-Init di `forward()`) + Inspeksi Anotasi `Healthy Leaf` (Tanpa Penghapusan Otomatis)

In [ ]:
import copy
import types
import torch
import torch.nn as nn
from ultralytics import YOLO
import ultralytics.nn.modules as ult_modules
import ultralytics.nn.tasks as ult_tasks

CLASS_COLORS = [
    (0, 140, 255),   # 0: Tea algal leaf spot - Orange
    (0, 0, 255),     # 1: Brown Blight        - Red
    (0, 215, 255),   # 2: Gray Blight         - Gold/Yellow
    (255, 0, 255),   # 3: Helopeltis          - Magenta
    (50, 50, 255),   # 4: Red Spider          - Coral Red
    (0, 255, 0),     # 5: Green Mirid Bug     - Green
    (255, 255, 0),   # 6: Healty Leaf         - Cyan
]

# ==============================================================================
# 1. COORDINATE ATTENTION: INISIALISASI PERMANEN DI __init__()
# ==============================================================================
class CoordAtt(nn.Module):
    """
    Coordinate Attention Block (Hou et al., CVPR 2021).
    Seluruh layer konvolusi dibuat SATU KALI di __init__(), tidak pernah dibuat ulang di forward().
    """
    def __init__(self, inp: int = 128, oup: int = None, reduction: int = 32):
        super().__init__()
        inp = int(inp)
        oup = int(oup) if oup is not None else inp
        reduction = max(1, int(reduction))

        self.inp = inp
        self.oup = oup
        self.reduction = reduction
        self.mip = max(8, inp // reduction)

        self.pool_h = nn.AdaptiveAvgPool2d((None, 1))
        self.pool_w = nn.AdaptiveAvgPool2d((1, None))
        self.conv1  = nn.Conv2d(inp, self.mip, kernel_size=1, stride=1, padding=0, bias=False)
        self.bn1    = nn.BatchNorm2d(self.mip)
        self.act    = nn.Hardswish()
        self.conv_h = nn.Conv2d(self.mip, oup, kernel_size=1, stride=1, padding=0, bias=True)
        self.conv_w = nn.Conv2d(self.mip, oup, kernel_size=1, stride=1, padding=0, bias=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, C, H, W = x.shape
        if C != self.inp:
            raise RuntimeError(
                f"[CoordAtt Error] Channel input ({C}) != inisialisasi CoordAtt (inp={self.inp})."
            )
        identity = x
        x_h = self.pool_h(x)
        x_w = self.pool_w(x).permute(0, 1, 3, 2)
        y = torch.cat([x_h, x_w], dim=2)
        y = self.act(self.bn1(self.conv1(y)))
        x_h_enc, x_w_enc = torch.split(y, [H, W], dim=2)
        x_w_enc = x_w_enc.permute(0, 1, 3, 2)
        a_h = self.conv_h(x_h_enc).sigmoid()
        a_w = self.conv_w(x_w_enc).sigmoid()
        return identity * a_h * a_w


class CoordAttYOLO(CoordAtt):
    def __init__(self, c1: int = 128, c2: int = 32, reduction: int = None, *args, **kwargs):
        if isinstance(c1, (list, tuple)):
            c1 = c1[0]
        if "inp" in kwargs:
            c1 = kwargs["inp"]
        if "reduction" in kwargs and kwargs["reduction"] is not None:
            red = int(kwargs["reduction"])
            oup = int(c1)
        elif reduction is not None:
            oup = int(c2)
            red = int(reduction)
        else:
            oup = int(c1)
            red = int(c2) if int(c2) <= 64 else 32
        super().__init__(inp=int(c1), oup=oup, reduction=red)


def register_ca_everywhere():
    setattr(nn, "CoordAtt", CoordAtt)
    setattr(nn, "CoordAttYOLO", CoordAttYOLO)
    setattr(ult_modules, "CoordAtt", CoordAtt)
    setattr(ult_modules, "CoordAttYOLO", CoordAttYOLO)
    setattr(ult_tasks, "CoordAtt", CoordAtt)
    setattr(ult_tasks, "CoordAttYOLO", CoordAttYOLO)
    ult_tasks.__dict__["CoordAtt"] = CoordAtt
    ult_tasks.__dict__["CoordAttYOLO"] = CoordAttYOLO
    if hasattr(ult_tasks, "parse_model"):
        ult_tasks.parse_model.__globals__["CoordAtt"] = CoordAtt
        ult_tasks.parse_model.__globals__["CoordAttYOLO"] = CoordAttYOLO
    curr_mod = sys.modules[__name__]
    for pkg in ("phase3_yolo_ca", "phase3_yolo_ca_finetune", "phase3_scripts"):
        if pkg not in sys.modules:
            p_obj = types.ModuleType(pkg)
            p_obj.ca_module = curr_mod
            sys.modules[pkg] = p_obj
        else:
            setattr(sys.modules[pkg], "ca_module", curr_mod)
        sys.modules[f"{pkg}.ca_module"] = curr_mod

register_ca_everywhere()

# ==============================================================================
# 2. REVISI CELL 4 (Prioritas 3): INSPEKSI ANOTASI 'HEALTHY LEAF' TANPA PENGHAPUSAN OTOMATIS
# ==============================================================================
def _box_iou_norm(b1, b2) -> float:
    """Menghitung IoU antara dua box YOLO ternormalisasi [xc, yc, bw, bh]."""
    x1_a, y1_a, x2_a, y2_a = b1[0] - b1[2]/2, b1[1] - b1[3]/2, b1[0] + b1[2]/2, b1[1] + b1[3]/2
    x1_b, y1_b, x2_b, y2_b = b2[0] - b2[2]/2, b2[1] - b2[3]/2, b2[0] + b2[2]/2, b2[1] + b2[3]/2
    ix1, iy1 = max(x1_a, x1_b), max(y1_a, y1_b)
    ix2, iy2 = min(x2_a, x2_b), min(y2_a, y2_b)
    inter = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)
    a1 = max(0.0, b1[2] * b1[3])
    a2 = max(0.0, b2[2] * b2[3])
    return inter / max(a1 + a2 - inter, 1e-6)

def audit_annotations_and_healthy_leaf(data_dir: str, auto_delete_healthy_on_diseased: bool = False):
    """
    REVISI CELL 4:
    - Menonaktifkan penghapusan otomatis label 'Healthy Leaf' (auto_delete_healthy_on_diseased=False)
      ketika terdapat bercak penyakit dalam gambar yang sama (karena pada foto multi-daun,
      satu daun bisa sehat sementara daun di sebelahnya berpenyakit).
    - Menyimpan daftar seluruh berkas yang memiliki ko-okurensi 'Healthy Leaf' + penyakit ke dalam
      'cooccurring_healthy_and_disease_inspection.csv' beserta nilai IoU spasial antar kotak
      agar peneliti dapat memeriksa anotasinya terlebih dahulu.
    """
    stats = {
        c: {"images": set(), "boxes": 0, "areas": [], "small_boxes": 0, "full_leaf_boxes": 0}
        for c in range(len(CLASS_NAMES))
    }
    cooccurrence_records = []

    for sp in ("train", "val", "test"):
        lbl_dir = os.path.join(data_dir, "labels", sp)
        if not os.path.exists(lbl_dir):
            continue
        for lbl_p in sorted(glob.glob(os.path.join(lbl_dir, "*.txt"))):
            bname = os.path.basename(lbl_p)
            if bname.startswith("bal_"):
                continue
            with open(lbl_p, "r", encoding="utf-8") as f:
                raw_lines = [ln.strip() for ln in f if ln.strip()]
            if not raw_lines:
                continue

            parsed_boxes = []
            for ln in raw_lines:
                parts = ln.split()
                if len(parts) < 5:
                    continue
                cid = int(parts[0])
                xc, yc, bw, bh = [float(x) for x in parts[1:5]]
                parsed_boxes.append((cid, xc, yc, bw, bh))

            healthy_boxes = [b for b in parsed_boxes if b[0] == 6]
            disease_boxes = [b for b in parsed_boxes if 0 <= b[0] < 6]

            if healthy_boxes and disease_boxes:
                # Hitung IoU maksimum antara kotak Healthy Leaf dan kotak bercak penyakit di citra tersebut
                max_overlap_iou = max(
                    _box_iou_norm(hb[1:], db[1:])
                    for hb in healthy_boxes for db in disease_boxes
                )
                cooccurrence_records.append({
                    "split": sp,
                    "label_file": bname,
                    "num_healthy_boxes": len(healthy_boxes),
                    "num_disease_boxes": len(disease_boxes),
                    "disease_classes": ", ".join(sorted({CLASS_NAMES[db[0]] for db in disease_boxes})),
                    "max_spatial_iou": round(max_overlap_iou, 4),
                    "interpretation": "OVERLAP_DAUN_SAMA (Perlu Cek Manual)" if max_overlap_iou >= 0.30 else "DAUN_BERBEDA_DALAM_SATU_FOTO (Valid)",
                })
                if auto_delete_healthy_on_diseased:
                    parsed_boxes = disease_boxes
                    with open(lbl_p, "w", encoding="utf-8") as f:
                        f.write("\n".join(f"{c} {x:.6f} {y:.6f} {w:.6f} {h:.6f}" for c, x, y, w, h in parsed_boxes) + "\n")

            for cid, xc_c, yc_c, bw_c, bh_c in parsed_boxes:
                if cid in stats:
                    area = bw_c * bh_c
                    stats[cid]["images"].add(f"{sp}/{bname}")
                    stats[cid]["boxes"] += 1
                    stats[cid]["areas"].append(area)
                    if area < 0.05:
                        stats[cid]["small_boxes"] += 1
                    if area >= 0.50:
                        stats[cid]["full_leaf_boxes"] += 1

    audit_rows = []
    for cid, cname in enumerate(CLASS_NAMES):
        st = stats[cid]
        n_b = max(1, st["boxes"])
        med_area = float(np.median(st["areas"])) * 100.0 if st["areas"] else 0.0
        audit_rows.append({
            "ID": cid,
            "Kelas": cname,
            "Citra Unik": len(st["images"]),
            "Total Box": st["boxes"],
            "Median Area Box (%)": f"{med_area:.2f}%",
            "Bercak Kecil (<5% Area)": f"{st['small_boxes']} ({100.0*st['small_boxes']/n_b:.1f}%)",
            "Full-Leaf Box (>=50% Area)": f"{st['full_leaf_boxes']} ({100.0*st['full_leaf_boxes']/n_b:.1f}%)",
        })

    df_audit = pd.DataFrame(audit_rows)
    df_cooc  = pd.DataFrame(cooccurrence_records)
    cooc_csv_path = os.path.join(BASE_WORK_DIR, "cooccurring_healthy_and_disease_inspection.csv")
    df_cooc.to_csv(cooc_csv_path, index=False)

    print("\n" + "=" * 94)
    print("  [AUDIT ANOTASI & KELAS HEALTHY LEAF] (MODE BACA-SAJA: TIDAK ADA PENGHAPUSAN OTOMATIS)")
    print("=" * 94)
    print(df_audit.to_string(index=False))
    print(f"  * Berkas dengan ko-okurensi 'Healthy Leaf' + Bercak Penyakit : {len(df_cooc)} berkas (0 dihapus otomatis)")
    print(f"  * Laporan inspeksi disimpan di                               : {cooc_csv_path}")
    if not df_cooc.empty:
        print("\n  [SAMPEL INSPEKSI KO-OKURENSI HEALTHY LEAF + PENYAKIT]:")
        print(df_cooc.head(8).to_string(index=False))
    return df_audit, df_cooc

_, _ = audit_annotations_and_healthy_leaf(SINGLE_LEAF_DIR, auto_delete_healthy_on_diseased=False)

# ==============================================================================
# 3. PENYEIMBANGAN KELAS MINORITAS PADA SPLIT 'TRAIN' (TERKENDALI & ANTI-BOCOR)
# ==============================================================================
def balance_minority_train_split(
    data_dir: str,
    target_min: int = 650,
    max_dup_per_image: int = 3,
    seed: int = 42,
):
    img_tr = os.path.join(data_dir, "images", "train")
    lbl_tr = os.path.join(data_dir, "labels", "train")

    for old_f in glob.glob(os.path.join(img_tr, "bal_*")) + glob.glob(os.path.join(lbl_tr, "bal_*")):
        os.remove(old_f)
    for cache_f in glob.glob(os.path.join(data_dir, "labels", "*.cache")):
        os.remove(cache_f)

    cls_pure_files = {c: [] for c in range(len(CLASS_NAMES))}
    cls_inst_counts = {c: 0 for c in range(len(CLASS_NAMES))}

    for lbl_p in sorted(glob.glob(os.path.join(lbl_tr, "*.txt"))):
        bname = os.path.splitext(os.path.basename(lbl_p))[0]
        if bname.startswith(("bal_", "lapangan_", "rf_", "field_", "vfield_")):
            continue
        with open(lbl_p, "r", encoding="utf-8") as f:
            lines = [ln.strip() for ln in f if ln.strip()]
        if not lines:
            continue
        cids_in_file = []
        for ln in lines:
            ps = ln.split()
            if len(ps) >= 5:
                cid = int(ps[0])
                if cid in cls_inst_counts:
                    cls_inst_counts[cid] += 1
                    cids_in_file.append(cid)
        if not cids_in_file:
            continue
        unique_cids = set(cids_in_file)
        if len(unique_cids) == 1:
            cid_only = next(iter(unique_cids))
            for ext in (".jpg", ".jpeg", ".png", ".JPG", ".PNG"):
                ip = os.path.join(img_tr, bname + ext)
                if os.path.exists(ip):
                    cls_pure_files[cid_only].append((ip, lbl_p, len(cids_in_file)))
                    break

    rng = random.Random(seed)
    added_files = 0
    print("\n" + "=" * 94)
    print("  [PENYEIMBANGAN KELAS MINORITAS SPLIT TRAIN] (VAL & TEST 100% TIDAK DISENTUH)")
    print("=" * 94)
    for cid in range(len(CLASS_NAMES)):
        orig_list = cls_pure_files[cid]
        n_orig_imgs = len(orig_list)
        n_orig_inst = cls_inst_counts[cid]
        if 0 < n_orig_imgs < target_min:
            max_allowed_add = n_orig_imgs * (max_dup_per_image - 1)
            need = min(target_min - n_orig_imgs, max_allowed_add)
            picks = [orig_list[i % n_orig_imgs] for i in range(need)]
            rng.shuffle(picks)
            added_inst_c = 0
            for k, (src_img, src_lbl, n_b) in enumerate(picks, 1):
                ext = os.path.splitext(src_img)[1]
                stem = os.path.splitext(os.path.basename(src_img))[0]
                shutil.copy2(src_img, os.path.join(img_tr, f"bal_{cid}_{k:04d}_{stem}{ext}"))
                shutil.copy2(src_lbl, os.path.join(lbl_tr, f"bal_{cid}_{k:04d}_{stem}.txt"))
                added_files += 1
                added_inst_c += n_b
            print(f"  - Kelas {cid} ({CLASS_NAMES[cid]:<20}): Citra {n_orig_imgs:4d} -> {n_orig_imgs + need:4d} (+{need:3d}) | Box: {n_orig_inst:4d} -> {n_orig_inst + added_inst_c:4d}")
        else:
            print(f"  - Kelas {cid} ({CLASS_NAMES[cid]:<20}): Citra {n_orig_imgs:4d} -> {n_orig_imgs:4d} (tetap) | Box: {n_orig_inst:4d}")

balance_minority_train_split(SINGLE_LEAF_DIR, target_min=650, max_dup_per_image=3, seed=42)


### [Cell 6 | Prioritas 3] Arsitektur `yolo26n_ca.yaml` & Pemeriksaan Detail Jumlah dan Kesesuaian Bobot Pretrained

In [ ]:
MODEL_YAML = os.path.join(BASE_WORK_DIR, "yolo26n_ca.yaml")
YAML_CONTENT = """nc: 7
end2end: True
reg_max: 1

scales:
  n: [0.50, 0.25, 1024]

backbone:
  - [-1, 1, Conv, [64, 3, 2]]             # 0 - P1/2
  - [-1, 1, Conv, [128, 3, 2]]            # 1 - P2/4
  - [-1, 2, C3k2, [256, False, 0.25]]     # 2 - P2/4
  - [-1, 1, Conv, [256, 3, 2]]            # 3 - P3/8
  - [-1, 2, C3k2, [512, False, 0.25]]     # 4 - P3/8 (out_ch=128)
  - [-1, 1, CoordAttYOLO, [128, 32]]      # 5 - CA-P3 (out_ch=128)

  - [-1, 1, Conv, [512, 3, 2]]            # 6 - P4/16
  - [-1, 2, C3k2, [512, True]]            # 7 - P4/16 (out_ch=128)
  - [-1, 1, CoordAttYOLO, [128, 32]]      # 8 - CA-P4 (out_ch=128)

  - [-1, 1, Conv, [1024, 3, 2]]           # 9 - P5/32
  - [-1, 2, C3k2, [1024, True]]           # 10 - P5/32
  - [-1, 1, SPPF, [1024, 5, 3, True]]     # 11 - P5/32
  - [-1, 2, C2PSA, [1024]]                # 12 - P5/32 (out_ch=256)
  - [-1, 1, CoordAttYOLO, [256, 32]]      # 13 - CA-P5 (out_ch=256)

head:
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]] # 14
  - [[-1, 8], 1, Concat, [1]]                  # 15 cat P4 CA
  - [-1, 2, C3k2, [512, True]]                 # 16

  - [-1, 1, nn.Upsample, [None, 2, "nearest"]] # 17
  - [[-1, 5], 1, Concat, [1]]                  # 18 cat P3 CA
  - [-1, 2, C3k2, [256, True]]                 # 19

  - [-1, 1, Conv, [256, 3, 2]]                 # 20
  - [[-1, 16], 1, Concat, [1]]                 # 21 cat head P4
  - [-1, 2, C3k2, [512, True]]                 # 22

  - [-1, 1, Conv, [512, 3, 2]]                 # 23
  - [[-1, 13], 1, Concat, [1]]                 # 24 cat P5 CA
  - [-1, 1, C3k2, [1024, True, 0.5, True]]     # 25

  - [[19, 22, 25], 1, Detect, [nc]]            # 26 Detect(P3, P4, P5)
"""
with open(MODEL_YAML, "w", encoding="utf-8") as f:
    f.write(YAML_CONTENT)

REMAPPED_PT = os.path.join(BASE_WORK_DIR, "yolo26n_ca_coco_init.pt")
LAYER_MAP = {
    0: 0, 1: 1, 2: 2, 3: 3, 4: 4,
    5: 6, 6: 7, 7: 9, 8: 10, 9: 11, 10: 12,
    11: 14, 12: 15, 13: 16, 14: 17, 15: 18, 16: 19,
    17: 20, 18: 21, 19: 22, 20: 23, 21: 24, 22: 25, 23: 26
}

def remap_and_verify_yolo26n_to_ca(base_pt: str, yaml_path: str, out_pt: str):
    """
    REVISI CELL 6 (Prioritas 3):
    1. Memeriksa secara rinci jumlah dan persentase kesesuaian bobot pretrained per blok layer:
       - Memastikan 100% tensor Backbone & Neck (23 layer: indeks lama 0..22) berhasil dimuat
         tanpa satu pun shape mismatch.
       - Memeriksa jumlah tensor pada Detect Head (layer 26) yang berhasil dimuat vs diadaptasi (80->7 kelas).
    2. Memverifikasi kompatibilitas dimensi channel antara layer produsen fitur (layer 4, 7, 12)
       dan blok CoordAttYOLO (layer 5, 8, 13) melalui hook pemeriksaan tensor aktual.
    """
    register_ca_everywhere()
    ca_yolo = YOLO(yaml_path)
    ca_model = ca_yolo.model
    ca_sd = ca_model.state_dict()

    ckpt = torch.load(base_pt, map_location="cpu", weights_only=False)
    base_sd = (ckpt.get("model") or ckpt.get("ema")).float().state_dict()

    new_sd = copy.deepcopy(ca_sd)
    backbone_neck_total, backbone_neck_matched = 0, 0
    head_total, head_matched, head_adapted = 0, 0, 0
    unmapped_base_keys = []

    for old_k, tensor in base_sd.items():
        parts = old_k.split(".")
        if len(parts) >= 3 and parts[0] == "model" and parts[1].isdigit():
            old_idx = int(parts[1])
            new_idx = LAYER_MAP.get(old_idx)
            if new_idx is None:
                unmapped_base_keys.append(old_k)
                continue
            new_k = ".".join(["model", str(new_idx)] + parts[2:])
            is_detect_head = (old_idx == 23)
            if is_detect_head:
                head_total += 1
                if new_k in new_sd and new_sd[new_k].shape == tensor.shape:
                    new_sd[new_k] = tensor.to(dtype=new_sd[new_k].dtype)
                    head_matched += 1
                else:
                    head_adapted += 1
            else:
                backbone_neck_total += 1
                if new_k in new_sd and new_sd[new_k].shape == tensor.shape:
                    new_sd[new_k] = tensor.to(dtype=new_sd[new_k].dtype)
                    backbone_neck_matched += 1

    # Wajib 100% cocok untuk seluruh layer Backbone & Neck!
    assert backbone_neck_matched == backbone_neck_total and backbone_neck_total > 0, (
        f"KRITIS: Bobot Backbone/Neck tidak cocok 100% ({backbone_neck_matched}/{backbone_neck_total})!"
    )
    assert len(unmapped_base_keys) == 0, f"Ada layer pretrained yang tidak terpetakan: {unmapped_base_keys}"

    ca_model.load_state_dict(new_sd, strict=True)

    # Verifikasi kompatibilitas channel aktual dengan forward hook pada 3 blok CoordAttYOLO
    ca_layers = [(idx, m) for idx, m in enumerate(ca_model.model) if isinstance(m, CoordAtt)]
    assert len(ca_layers) == 3, f"Seharusnya terdapat 3 blok CoordAtt, ditemukan {len(ca_layers)}!"
    param_ids_before = [id(p) for _, m in ca_layers for p in m.parameters()]
    observed_shapes = {}
    hooks = []
    for idx, m in ca_layers:
        def _make_hook(layer_idx):
            def _hk(mod, inp_t, out_t):
                observed_shapes[layer_idx] = (tuple(inp_t[0].shape), tuple(out_t.shape))
            return _hk
        hooks.append(m.register_forward_hook(_make_hook(idx)))

    ca_model.eval()
    with torch.no_grad():
        for test_sz in (640, 768):
            _ = ca_model(torch.zeros(1, 3, test_sz, test_sz))
    for h in hooks:
        h.remove()

    param_ids_after = [id(p) for _, m in ca_layers for p in m.parameters()]
    assert param_ids_before == param_ids_after, "KRITIS: Parameter CoordAtt berubah saat forward()!"

    ca_tensor_count = sum(len(m.state_dict()) for _, m in ca_layers)
    total_matched = backbone_neck_matched + head_matched

    save_obj = {
        "epoch": -1,
        "best_fitness": None,
        "model": copy.deepcopy(ca_model).half(),
        "ema": None,
        "updates": None,
        "optimizer": None,
        "train_args": ckpt.get("train_args") if isinstance(ckpt.get("train_args"), dict) else {},
        "date": None,
        "version": ckpt.get("version", "8.4.178"),
    }
    torch.save(save_obj, out_pt)

    print("=" * 88)
    print("  [PEMERIKSAAN BOBOT PRETRAINED & KOMPATIBILITAS ARSITEKTUR YOLO26n + CA]")
    print("=" * 88)
    print(f"  1. Backbone & Neck (23 Layer) : {backbone_neck_matched}/{backbone_neck_total} tensor cocok (100.00% Pretrained Loaded)")
    print(f"  2. Detect Head (Layer 26)     : {head_matched}/{head_total} tensor box/DFL cocok | {head_adapted} tensor cls (80->7 kelas)")
    print(f"  3. Modul CoordAtt (Layer 5,8,13): {ca_tensor_count} tensor baru terinisialisasi & terverifikasi")
    print(f"  4. Total Pretrained Termuat   : {total_matched}/{len(base_sd)} tensor ({100.0*total_matched/len(base_sd):.2f}%)")
    for idx, m in ca_layers:
        in_s, out_s = observed_shapes.get(idx, (None, None))
        print(f"     - Layer {idx:2d} ({m.__class__.__name__}): inp={m.inp}, oup={m.oup}, mip={m.mip} | Tensor 768px: {in_s} -> {out_s}")
    print(f"  5. Status Konsistensi Parameter: LULUS (ID parameter tetap sebelum & sesudah forward) -> {out_pt}")

remap_and_verify_yolo26n_to_ca(BASE_DET_PT, MODEL_YAML, REMAPPED_PT)


### [Cell 8 | Dipertahankan + Smoke Test] Smoke Test Gradien Arsitektur CA & Training Model Awal YOLO26n + CA (Fokus Bercak Kecil)

In [ ]:
# ==============================================================================
# REVISI CELL 8 (Tambahan Smoke Test Sebelum Training Penuh):
# Menjalankan Smoke Test 1-batch (Forward + Backward Pass) pada model YOLO26n+CA
# untuk membuktikan bahwa seluruh layer CoordAttYOLO menerima gradien non-nol
# dan terhubung sempurna ke grafik komputasi PyTorch sebelum training 50 epoch dimulai.
# ==============================================================================
def run_ca_architecture_smoke_test(yaml_path: str, weights_path: str, imgsz_test: int = 640):
    register_ca_everywhere()
    test_yolo = YOLO(yaml_path)
    ckpt = torch.load(weights_path, map_location="cpu", weights_only=False)
    test_yolo.model.load_state_dict(ckpt["model"].float().state_dict(), strict=True)
    net = test_yolo.model.train()
    for p in net.parameters():
        p.requires_grad_(True)

    dummy_x = torch.randn(2, 3, imgsz_test, imgsz_test)
    preds = net(dummy_x)
    # Hitung scalar surrogate loss dari seluruh tensor keluaran head untuk menguji aliran gradien balik
    def _sum_tensors(obj):
        if isinstance(obj, torch.Tensor):
            return obj.float().pow(2).mean()
        if isinstance(obj, (list, tuple)):
            return sum(_sum_tensors(x) for x in obj)
        if isinstance(obj, dict):
            return sum(_sum_tensors(v) for v in obj.values())
        return 0.0

    loss_smoke = _sum_tensors(preds)
    loss_smoke.backward()

    ca_grad_norms = {}
    for idx, m in enumerate(net.model):
        if isinstance(m, CoordAtt):
            g_norm = sum(float(p.grad.abs().sum().item()) for p in m.parameters() if p.grad is not None)
            ca_grad_norms[idx] = g_norm
            assert g_norm > 0.0 and np.isfinite(g_norm), f"Gradien pada CoordAtt layer {idx} bernilai 0 atau NaN!"

    print("[SMOKE TEST LULUS] Aliran Forward & Backward Gradient pada 3 blok CoordAttYOLO terverifikasi:")
    for idx, gn in ca_grad_norms.items():
        print(f"  - CoordAtt Layer {idx:2d} : Total |grad| = {gn:.6e} (Aktif & Terdaftar di Autograd)")
    del net, test_yolo

run_ca_architecture_smoke_test(MODEL_YAML, REMAPPED_PT, imgsz_test=640)

EPOCHS = 50
IMGSZ  = 768
BATCH  = 16
RUN_NAME = "yolo26n_ca_cls_opt_50ep"
PROJECT_RUNS = os.path.join(BASE_WORK_DIR, "phase3_cls_opt_runs")

init_weights = REMAPPED_PT
print(f"\n[INFO] Memuat bobot awal YOLO26n+CA (Pretrained Remapped): {init_weights}")

model = YOLO(MODEL_YAML)

train_kwargs = dict(
    data=SINGLE_LEAF_YAML,
    pretrained=init_weights,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    workers=8,
    device="cuda" if torch.cuda.is_available() else "cpu",
    project=PROJECT_RUNS,
    name=RUN_NAME,
    exist_ok=True,
    patience=15,
    save_period=10,
    seed=42,
    cos_lr=True,
    box=8.5,
    cls=1.25,
    dfl=1.5,
    hsv_h=0.004,
    hsv_s=0.35,
    hsv_v=0.35,
    degrees=30.0,
    flipud=0.5,
    fliplr=0.5,
    scale=0.45,
    mosaic=1.0,
    copy_paste=0.15,
    close_mosaic=15,
    verbose=True,
    plots=True,
)

print(f"[INFO] Memulai training {RUN_NAME} ({EPOCHS} Epoch, {IMGSZ}x{IMGSZ}) pada Dataset Single Leaf...")
train_results = model.train(**train_kwargs)
BEST_WEIGHTS = os.path.join(PROJECT_RUNS, RUN_NAME, "weights", "best.pt")
print(f"[SUKSES] Pelatihan selesai! Bobot terbaik disimpan di: {BEST_WEIGHTS}")


### [Cell 10 | Prioritas 3] Evaluasi Model Terpisah (Single Leaf vs Lapangan) dengan Verifikasi Kelengkapan Ground Truth (Termasuk Gambar Tanpa Penyakit)

In [ ]:
import pandas as pd
from IPython.display import display

def audit_ground_truth_completeness(data_dir: str, split: str = "test") -> dict:
    """
    REVISI CELL 10 & CELL 26 (Prioritas 3):
    Memeriksa kelengkapan Ground Truth manual pada seluruh gambar di suatu split:
      - positive_gt_files : Berkas .txt ada dan berisi >= 1 anotasi bercak penyakit.
      - negative_gt_files : Berkas .txt ada namun kosong (0 baris) -> VALID sebagai Ground Truth
                            untuk gambar daun sehat / tanpa penyakit (negative sample).
      - missing_gt_files  : Gambar belum memiliki berkas .txt sama sekali (belum dianotasi).
    Evaluasi hanya sah dijalankan jika seluruh citra memiliki berkas Ground Truth (missing_gt_files == 0).
    """
    img_dir = os.path.join(data_dir, "images", split)
    lbl_dir = os.path.join(data_dir, "labels", split)
    if not os.path.exists(img_dir):
        return {"total_images": 0, "positive_gt": 0, "negative_empty_gt": 0, "missing_gt": 0, "is_complete": False}

    imgs = sorted([
        p for p in glob.glob(os.path.join(img_dir, "*.*"))
        if p.lower().endswith((".jpg", ".jpeg", ".png"))
    ])
    pos_gt, neg_gt, miss_gt = 0, 0, 0
    for img_p in imgs:
        stem = os.path.splitext(os.path.basename(img_p))[0]
        lbl_p = os.path.join(lbl_dir, stem + ".txt")
        if not os.path.exists(lbl_p):
            miss_gt += 1
        else:
            with open(lbl_p, "r", encoding="utf-8") as f:
                has_box = any(ln.strip() for ln in f)
            if has_box:
                pos_gt += 1
            else:
                neg_gt += 1

    total_imgs = len(imgs)
    is_complete = (total_imgs > 0) and (miss_gt == 0)
    return {
        "total_images": total_imgs,
        "positive_gt": pos_gt,
        "negative_empty_gt": neg_gt,
        "total_labeled_gt": pos_gt + neg_gt,
        "missing_gt": miss_gt,
        "is_complete": is_complete,
    }

def evaluate_model_on_dataset(
    weights_path: str,
    data_yaml_path: str,
    data_dir_path: str,
    split_name: str,
    eval_tag: str,
    imgsz: int = 768,
):
    gt_status = audit_ground_truth_completeness(data_dir_path, split=split_name)
    print(
        f"[AUDIT GT {eval_tag} ({split_name})] Total Citra: {gt_status['total_images']} | "
        f"Berpenyakit (Positif): {gt_status['positive_gt']} | "
        f"Tanpa Penyakit (Negatif Valid): {gt_status['negative_empty_gt']} | "
        f"Belum Berlabel (Missing): {gt_status['missing_gt']}"
    )
    if not gt_status["is_complete"]:
        raise RuntimeError(
            f"[EVALUASI DIBATALKAN] Split '{split_name}' pada '{data_dir_path}' belum memiliki Ground Truth lengkap! "
            f"Masih ada {gt_status['missing_gt']}/{gt_status['total_images']} citra tanpa berkas label .txt. "
            f"(Catatan: untuk citra tanpa penyakit, buat berkas .txt kosong sebagai Ground Truth negatif)."
        )

    register_ca_everywhere()
    eval_model = YOLO(weights_path)
    metrics = eval_model.val(
        data=data_yaml_path,
        split=split_name,
        imgsz=imgsz,
        batch=16,
        device="cuda" if torch.cuda.is_available() else "cpu",
        project=os.path.join(PROJECT_RUNS, RUN_NAME),
        name=eval_tag,
        exist_ok=True,
        save_json=True,
        plots=True,
        verbose=False,
    )

    mp = float(metrics.box.mp)
    mr = float(metrics.box.mr)
    map50 = float(metrics.box.map50)
    map5095 = float(metrics.box.map)
    f1 = 2 * mp * mr / (mp + mr + 1e-9)

    rows = []
    if hasattr(metrics.box, "ap_class_index") and metrics.box.ap_class_index is not None:
        for i, c_idx in enumerate(metrics.box.ap_class_index):
            cname = CLASS_NAMES[int(c_idx)] if int(c_idx) < len(CLASS_NAMES) else f"Class {c_idx}"
            p_c = float(metrics.box.p[i])
            r_c = float(metrics.box.r[i])
            ap50 = float(metrics.box.ap50[i])
            ap95 = float(metrics.box.ap[i]) if hasattr(metrics.box, "ap") else ap50 * 0.95
            f1_c = 2 * p_c * r_c / (p_c + r_c + 1e-9)
            rows.append({
                "Kelas": cname,
                "Precision": f"{p_c:.4f}",
                "Recall": f"{r_c:.4f}",
                "F1-Score": f"{f1_c:.4f}",
                "mAP@0.50": f"{ap50:.4f}",
                "mAP@0.50:0.95": f"{ap95:.4f}",
            })

    rows.append({
        "Kelas": "== MEAN / OVERALL ==",
        "Precision": f"{mp:.4f}",
        "Recall": f"{mr:.4f}",
        "F1-Score": f"{f1:.4f}",
        "mAP@0.50": f"{map50:.4f}",
        "mAP@0.50:0.95": f"{map5095:.4f}",
    })

    summary_dict = {
        "eval_tag": eval_tag,
        "weights": weights_path,
        "data_yaml": data_yaml_path,
        "split": split_name,
        "gt_completeness": gt_status,
        "precision": mp,
        "recall": mr,
        "f1": f1,
        "map50": map50,
        "map50_95": map5095,
    }
    out_json = os.path.join(PROJECT_RUNS, RUN_NAME, eval_tag, "summary_metrics.json")
    os.makedirs(os.path.dirname(out_json), exist_ok=True)
    with open(out_json, "w", encoding="utf-8") as f:
        json.dump(summary_dict, f, indent=2)

    return pd.DataFrame(rows), summary_dict

# 1. Verifikasi & Evaluasi pada Dataset Single Leaf (val & test wajib lengkap 100%)
_ = audit_ground_truth_completeness(SINGLE_LEAF_DIR, split="val")
df_single_leaf_eval, BASELINE_SINGLE_LEAF_METRICS = evaluate_model_on_dataset(
    weights_path=BEST_WEIGHTS,
    data_yaml_path=SINGLE_LEAF_YAML,
    data_dir_path=SINGLE_LEAF_DIR,
    split_name="test",
    eval_tag="test_eval_single_leaf",
    imgsz=IMGSZ,
)
print("\n" + "=" * 88)
print("  [TABEL 10.A] HASIL EVALUASI PADA DATASET SINGLE LEAF (100% GROUND TRUTH LENGKAP)")
print("=" * 88)
display(df_single_leaf_eval)

# 2. Verifikasi & Evaluasi Terpisah pada Dataset Lapangan (field_val & field_test wajib lengkap 100%)
field_val_gt_stat  = audit_ground_truth_completeness(FIELD_500_DIR, split="val")
field_test_gt_stat = audit_ground_truth_completeness(FIELD_500_DIR, split="test")
if field_test_gt_stat["is_complete"]:
    df_field_baseline_eval, BASELINE_FIELD_METRICS = evaluate_model_on_dataset(
        weights_path=BEST_WEIGHTS,
        data_yaml_path=FIELD_500_YAML,
        data_dir_path=FIELD_500_DIR,
        split_name="test",
        eval_tag="test_eval_field_baseline",
        imgsz=IMGSZ,
    )
    print("\n" + "=" * 88)
    print(
        f"  [TABEL 10.B] HASIL EVALUASI BASELINE PADA DATASET LAPANGAN "
        f"({field_test_gt_stat['positive_gt']} Positif + {field_test_gt_stat['negative_empty_gt']} Negatif Tanpa Penyakit)"
    )
    print("=" * 88)
    display(df_field_baseline_eval)
else:
    BASELINE_FIELD_METRICS = None
    print("\n" + "=" * 88)
    print("  [PERINGATAN KELENGKAPAN GT LAPANGAN — TABEL 10.B DITUNDA]")
    print(f"  - Field Val  : {field_val_gt_stat['positive_gt']} positif, {field_val_gt_stat['negative_empty_gt']} negatif (kosong valid), {field_val_gt_stat['missing_gt']} belum berlabel.")
    print(f"  - Field Test : {field_test_gt_stat['positive_gt']} positif, {field_test_gt_stat['negative_empty_gt']} negatif (kosong valid), {field_test_gt_stat['missing_gt']} belum berlabel.")
    print("  Pastikan seluruh citra val & test lapangan memiliki file .txt Ground Truth manual (kosong untuk daun tanpa penyakit).")
    print("=" * 88)


### [Cell 12 | Dipertahankan] Visualisasi Kurva Training + Grafik Perbandingan Sebelum & Sesudah Fine-Tuning

In [ ]:
import csv
import matplotlib.pyplot as plt
from IPython.display import Image as IPyImage, display, Markdown

def load_yolo_results_csv(csv_path: str):
    data = {"epoch": [], "box_loss": [], "cls_loss": [], "precision": [], "recall": [], "map50": [], "map50_95": [], "f1": []}
    if not os.path.exists(csv_path):
        return data
    with open(csv_path, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for r in reader:
            clean = {k.strip(): v.strip() for k, v in r.items() if k}
            try:
                ep = int(float(clean.get("epoch", len(data["epoch"]) + 1)))
                bl = float(clean.get("train/box_loss", 0.0))
                cl = float(clean.get("train/cls_loss", 0.0))
                pr = float(clean.get("metrics/precision(B)", 0.0))
                rc = float(clean.get("metrics/recall(B)", 0.0))
                m50 = float(clean.get("metrics/mAP50(B)", 0.0))
                m95 = float(clean.get("metrics/mAP50-95(B)", 0.0))
                f1 = 2 * pr * rc / (pr + rc + 1e-9)
                data["epoch"].append(ep)
                data["box_loss"].append(bl)
                data["cls_loss"].append(cl)
                data["precision"].append(pr)
                data["recall"].append(rc)
                data["map50"].append(m50)
                data["map50_95"].append(m95)
                data["f1"].append(f1)
            except ValueError:
                continue
    return data

base_csv_path = os.path.join(PROJECT_RUNS, RUN_NAME, "results.csv")
base_hist = load_yolo_results_csv(base_csv_path)

if base_hist["epoch"]:
    fig, axes = plt.subplots(2, 3, figsize=(18, 9))
    fig.suptitle(f"Kurva Konvergensi Training Awal YOLO26n + CA ({len(base_hist['epoch'])} Epoch, {IMGSZ}x{IMGSZ})", fontsize=15, fontweight="bold")
    axes[0, 0].plot(base_hist["epoch"], base_hist["box_loss"], color="#1f77b4", lw=2); axes[0, 0].set_title("Box Loss", fontweight="bold"); axes[0, 0].grid(True, alpha=0.3)
    axes[0, 1].plot(base_hist["epoch"], base_hist["cls_loss"], color="#d62728", lw=2); axes[0, 1].set_title("Classification Loss", fontweight="bold"); axes[0, 1].grid(True, alpha=0.3)
    axes[0, 2].plot(base_hist["epoch"], base_hist["precision"], color="#2ca02c", lw=2, label="Precision"); axes[0, 2].plot(base_hist["epoch"], base_hist["recall"], color="#e377c2", lw=2, label="Recall"); axes[0, 2].legend(); axes[0, 2].set_title("Precision & Recall", fontweight="bold"); axes[0, 2].grid(True, alpha=0.3)
    axes[1, 0].plot(base_hist["epoch"], base_hist["map50"], color="#17becf", lw=2); axes[1, 0].set_title("mAP@0.50", fontweight="bold"); axes[1, 0].grid(True, alpha=0.3)
    axes[1, 1].plot(base_hist["epoch"], base_hist["map50_95"], color="#bcbd22", lw=2); axes[1, 1].set_title("mAP@0.50:0.95", fontweight="bold"); axes[1, 1].grid(True, alpha=0.3)
    axes[1, 2].plot(base_hist["epoch"], base_hist["f1"], color="#8c564b", lw=2); axes[1, 2].set_title("F1-Score", fontweight="bold"); axes[1, 2].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(PROJECT_RUNS, RUN_NAME, "convergence_curves_baseline.png"), dpi=200, bbox_inches="tight")
    plt.show()

ft_csv_path = os.path.join(PROJECT_RUNS, "yolo26n_ca_cls_opt_finetune", "results.csv")
ft_hist = load_yolo_results_csv(ft_csv_path)

if base_hist["epoch"] and ft_hist["epoch"]:
    fig, axes = plt.subplots(1, 3, figsize=(19, 5.5))
    fig.suptitle("Perbandingan Kurva & Capaian Metrik: Sebelum (Baseline) vs Sesudah Fine-Tuning (Lapangan)", fontsize=14, fontweight="bold")
    axes[0].plot(base_hist["epoch"], base_hist["map50"], "--", color="#1f77b4", lw=2, label="Baseline mAP@0.50")
    axes[0].plot(ft_hist["epoch"], ft_hist["map50"], "-", color="#2ca02c", lw=2.5, label="Fine-Tuned mAP@0.50")
    axes[0].plot(base_hist["epoch"], base_hist["map50_95"], ":", color="#ff7f0e", lw=2, label="Baseline mAP@0.50:0.95")
    axes[0].plot(ft_hist["epoch"], ft_hist["map50_95"], "-", color="#d62728", lw=2.2, label="Fine-Tuned mAP@0.50:0.95")
    axes[0].set_title("Evolusi mAP (Baseline vs Fine-Tuned)", fontweight="bold")
    axes[0].set_xlabel("Epoch"); axes[0].grid(True, alpha=0.3); axes[0].legend()

    axes[1].plot(base_hist["epoch"], base_hist["f1"], "--", color="#9467bd", lw=2, label="Baseline F1-Score")
    axes[1].plot(ft_hist["epoch"], ft_hist["f1"], "-", color="#17becf", lw=2.5, label="Fine-Tuned F1-Score")
    axes[1].set_title("Evolusi F1-Score", fontweight="bold")
    axes[1].set_xlabel("Epoch"); axes[1].grid(True, alpha=0.3); axes[1].legend()

    metric_labels = ["Precision", "Recall", "F1-Score", "mAP@0.50", "mAP@0.50:0.95"]
    b_best_i = int(np.argmax(base_hist["map50"]))
    f_best_i = int(np.argmax(ft_hist["map50"]))
    b_vals = [base_hist["precision"][b_best_i], base_hist["recall"][b_best_i], base_hist["f1"][b_best_i], base_hist["map50"][b_best_i], base_hist["map50_95"][b_best_i]]
    f_vals = [ft_hist["precision"][f_best_i], ft_hist["recall"][f_best_i], ft_hist["f1"][f_best_i], ft_hist["map50"][f_best_i], ft_hist["map50_95"][f_best_i]]
    x_pos = np.arange(len(metric_labels))
    w_bar = 0.36
    axes[2].bar(x_pos - w_bar/2, b_vals, width=w_bar, label="Sebelum Fine-Tuning", color="#4c78a8")
    axes[2].bar(x_pos + w_bar/2, f_vals, width=w_bar, label="Sesudah Fine-Tuning", color="#54a24b")
    axes[2].set_xticks(x_pos); axes[2].set_xticklabels(metric_labels, rotation=15)
    axes[2].set_ylim(0, 1.08); axes[2].set_title("Perbandingan Capaian Terbaik", fontweight="bold")
    axes[2].grid(True, axis="y", alpha=0.3); axes[2].legend()

    plt.tight_layout()
    comp_fig_path = os.path.join(PROJECT_RUNS, "comparison_before_after_finetuning.png")
    plt.savefig(comp_fig_path, dpi=200, bbox_inches="tight")
    plt.show()
else:
    print("[INFO] Grafik perbandingan Sebelum vs Sesudah Fine-Tuning akan otomatis muncul setelah Cell 24 dijalankan.")

display(Markdown("#### Confusion Matrix & PR Curve pada Held-Out Test Set Single Leaf:"))
for p_img in [
    f"{PROJECT_RUNS}/{RUN_NAME}/test_eval_single_leaf/confusion_matrix_normalized.png",
    f"{PROJECT_RUNS}/{RUN_NAME}/test_eval_single_leaf/BoxPR_curve.png"
]:
    if os.path.exists(p_img):
        display(IPyImage(p_img, width=700))


### [Cell 14 | Dipertahankan] Visualisasi Prediksi Multi-Bercak (Menampilkan Seluruh Bounding Box Penyakit)

In [ ]:
import random
import cv2
import numpy as np
import matplotlib.pyplot as plt

register_ca_everywhere()
infer_model = YOLO(BEST_WEIGHTS)

test_img_dir = os.path.join(SINGLE_LEAF_DIR, "images", "test")
test_lbl_dir = os.path.join(SINGLE_LEAF_DIR, "labels", "test")

all_test_imgs = sorted(glob.glob(os.path.join(test_img_dir, "*.*")))
random.seed(42)
sample_imgs = random.sample(all_test_imgs, min(12, len(all_test_imgs)))

PALETTE_BGR = [
    (255, 80, 0),    # 0 Tea algal leaf spot
    (40, 40, 220),   # 1 Brown Blight
    (160, 160, 160), # 2 Gray Blight
    (0, 165, 255),   # 3 Helopeltis (Kelas penyakit/hama bercak - disertakan penuh!)
    (180, 0, 180),   # 4 Red Spider
    (0, 210, 210),   # 5 Green Mirid Bug
    (40, 180, 40),   # 6 Healty Leaf (Healthy Leaf - level daun utuh)
]

def compute_box_iou_xyxy(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])
    inter = max(0.0, xB - xA) * max(0.0, yB - yA)
    areaA = max(0.0, boxA[2] - boxA[0]) * max(0.0, boxA[3] - boxA[1])
    areaB = max(0.0, boxB[2] - boxB[0]) * max(0.0, boxB[3] - boxB[1])
    return inter / (areaA + areaB - inter + 1e-9)

def load_all_gt_boxes(lbl_path: str, w: int, h: int):
    gt_boxes = []
    if not os.path.exists(lbl_path):
        return gt_boxes
    with open(lbl_path, "r", encoding="utf-8") as f:
        for ln in f:
            p = ln.strip().split()
            if len(p) >= 5:
                cid = int(float(p[0]))
                cx, cy, bw, bh = map(float, p[1:5])
                x1 = int((cx - bw / 2) * w)
                y1 = int((cy - bh / 2) * h)
                x2 = int((cx + bw / 2) * w)
                y2 = int((cy + bh / 2) * h)
                gt_boxes.append({"cls": cid, "xyxy": [x1, y1, x2, y2]})
    return gt_boxes

fig, axes = plt.subplots(3, 4, figsize=(22, 16))
axes = axes.flatten()

total_gt_boxes = 0
matched_gt_boxes = 0
total_pred_boxes = 0

for idx, img_path in enumerate(sample_imgs):
    img_bgr = cv2.imread(img_path)
    h, w = img_bgr.shape[:2]
    base_name = os.path.splitext(os.path.basename(img_path))[0]
    lbl_path = os.path.join(test_lbl_dir, base_name + ".txt")

    gt_list = load_all_gt_boxes(lbl_path, w, h)
    total_gt_boxes += len(gt_list)

    res = infer_model.predict(img_path, imgsz=IMGSZ, conf=0.25, iou=0.45, verbose=False)[0]
    pred_list = []
    if res.boxes is not None and len(res.boxes) > 0:
        for b in res.boxes:
            cid = int(b.cls.item())
            cf = float(b.conf.item())
            x1, y1, x2, y2 = map(int, b.xyxy[0].tolist())
            pred_list.append({"cls": cid, "conf": cf, "xyxy": [x1, y1, x2, y2]})
    total_pred_boxes += len(pred_list)

    canvas = img_bgr.copy()
    for g in gt_list:
        gx1, gy1, gx2, gy2 = g["xyxy"]
        cv2.rectangle(canvas, (gx1, gy1), (gx2, gy2), (255, 255, 255), 1, cv2.LINE_AA)

    for p in pred_list:
        px1, py1, px2, py2 = p["xyxy"]
        cid = p["cls"]
        cf = p["conf"]
        col = PALETTE_BGR[cid % len(PALETTE_BGR)]
        cv2.rectangle(canvas, (px1, py1), (px2, py2), col, 2)
        tag = f"{CLASS_NAMES[cid][:12]} {cf:.2f}"
        (tw, th), _ = cv2.getTextSize(tag, cv2.FONT_HERSHEY_SIMPLEX, 0.42, 1)
        ty = max(py1 - 4, th + 4)
        cv2.rectangle(canvas, (px1, ty - th - 4), (px1 + tw + 4, ty + 2), col, -1)
        cv2.putText(canvas, tag, (px1 + 2, ty - 2), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (255, 255, 255), 1, cv2.LINE_AA)

    used_pred = set()
    gt_hits = 0
    for g in gt_list:
        best_iou, best_j = 0.0, -1
        for j, p in enumerate(pred_list):
            if j in used_pred or p["cls"] != g["cls"]:
                continue
            iou_v = compute_box_iou_xyxy(g["xyxy"], p["xyxy"])
            if iou_v > best_iou:
                best_iou, best_j = iou_v, j
        if best_iou >= 0.50 and best_j >= 0:
            used_pred.add(best_j)
            gt_hits += 1
    matched_gt_boxes += gt_hits

    img_rgb = cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB)
    axes[idx].imshow(img_rgb)
    axes[idx].axis("off")
    axes[idx].set_title(
        f"GT Boxes: {len(gt_list)} | Pred Boxes: {len(pred_list)}\nMatched (IoU>=0.5): {gt_hits}/{len(gt_list)}",
        fontsize=9.5,
        fontweight="bold",
        color="green" if gt_hits == len(gt_list) else "#b85d00",
    )

recall_sample = (matched_gt_boxes / max(1, total_gt_boxes)) * 100
plt.suptitle(
    f"Visualisasi Multi-Bercak pada Held-Out Test Set\n"
    f"Total GT Boxes: {total_gt_boxes} | Total Pred Boxes: {total_pred_boxes} | Matched GT (IoU>=0.5): {matched_gt_boxes}/{total_gt_boxes} ({recall_sample:.1f}%)",
    fontsize=14,
    fontweight="bold",
)
plt.tight_layout()
out_vis = os.path.join(PROJECT_RUNS, RUN_NAME, "test_predictions_multilesion_grid.png")
plt.savefig(out_vis, dpi=200, bbox_inches="tight")
plt.show()


### [Cell 16 | Prioritas 2] Pipeline Deteksi Daun Lapangan (`YOLO26n-seg` sebagai Metode Utama, HSV sebagai Pembanding/Draft, & Status `No lesion detected`)

In [ ]:
# ==============================================================================
# REVISI CELL 16 (Prioritas 2 + Perbaikan Audit Poin A & Poin D):
# 1. Poin A (Kelas Bercak vs Healthy Leaf):
#    - TIDAK mengecualikan Kelas 3 (`Helopeltis`), karena Kelas 3 adalah penyakit/hama utama.
#    - Menyaring Kelas 6 (`HEALTHY_LEAF_CLASS_ID = 6`, `Healty Leaf`) dari `detect_lesions()`
#      ketika `exclude_healthy_leaf_class=True` karena kelas 6 merupakan anotasi daun utuh.
# 2. Poin D (Urutan Inisialisasi Segmentasi Daun):
#    - TIDAK memuat `yolo26n-seg.pt` (model COCO umum) sebagai pengganti model daun teh.
#    - `leaf_seg_model` hanya dimuat jika bobot hasil training khusus daun teh (`LEAF_SEG_WEIGHTS`
#      dari Cell 30) sudah tersedia. Sebelum Cell 30 dijalankan, `detect_lesions()` tetap
#      berfungsi 100% independen untuk keperluan pseudo-labeling di Cell 22.
# 3. Status daun tanpa deteksi bercak ditetapkan sebagai 'No lesion detected'.
# 4. Penanganan eksplisit bercak di antara dua daun (`AMBIGUOUS_OVERLAP`).
# ==============================================================================
from torchvision.ops import nms as torch_nms

LEAF_SEG_WEIGHTS = os.path.join(BASE_WORK_DIR, "phase3_cls_opt_runs", "yolo26n_seg_tea_leaf", "weights", "best.pt")

class TeaLeafFieldDetector:
    """
    Pipeline terpisah & modular untuk:
      1. detect_leaves()                     : Segmentasi setiap daun menggunakan YOLO26n-seg khusus daun teh (Cell 30).
      2. generate_hsv_draft_or_comparison_masks(): Metode HSV/ExG khusus pembanding / pembuat draft mask.
      3. detect_lesions()                    : Deteksi independen seluruh bercak penyakit (Kelas 0..5, termasuk Kelas 3 Helopeltis)
                                               menggunakan YOLO26n+CA, menyaring Kelas 6 ('Healty Leaf').
      4. associate_leaves_and_lesions()      : Menghubungkan bercak ke daun dengan deteksi overlap ambigu
                                               serta status 'No lesion detected' untuk daun tanpa bercak.
    """
    def __init__(self, lesion_weights: str, leaf_seg_weights: str = None, device: str = None):
        register_ca_everywhere()
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.lesion_model = YOLO(lesion_weights)
        self.leaf_seg_model = None
        self.leaf_seg_weights_path = None
        self.seg_mode = "WAITING_FOR_CELL_30_TRAINING (Belum dilatih — jalankan Cell 28 & 30 terlebih dahulu)"

        if leaf_seg_weights and os.path.exists(leaf_seg_weights):
            self.load_trained_leaf_seg(leaf_seg_weights)
        else:
            print(
                f"[INFO] TeaLeafFieldDetector siap untuk deteksi bercak independen (`detect_lesions`).\n"
                f"       Model segmentasi khusus daun teh belum tersedia di `{leaf_seg_weights}`.\n"
                f"       (Tidak memuat `yolo26n-seg.pt` COCO umum agar tidak menghasilkan mask palsu; jalankan Cell 28 & 30 untuk melatih YOLO26n-seg daun teh)."
            )

    def load_trained_leaf_seg(self, leaf_seg_weights: str):
        """Memuat bobot YOLO26n-seg yang telah dilatih khusus untuk instance daun teh pada Cell 30."""
        if not (leaf_seg_weights and os.path.exists(leaf_seg_weights)):
            raise FileNotFoundError(f"Bobot YOLO26n-seg khusus daun teh tidak ditemukan: {leaf_seg_weights}")
        self.leaf_seg_model = YOLO(leaf_seg_weights)
        self.leaf_seg_weights_path = leaf_seg_weights
        self.seg_mode = f"YOLO26n-seg (Trained Tea Leaf Instance Model: {os.path.basename(leaf_seg_weights)})"
        print(f"[OK] Model segmentasi daun teh berhasil dimuat: {leaf_seg_weights}")

    @staticmethod
    def generate_hsv_draft_or_comparison_masks(img_bgr: np.ndarray, min_area_ratio: float = 0.008) -> list:
        """
        Metode Pembanding / Pembuat Draft Mask (HSV + Excess Green + Watershed):
        TIDAK digunakan sebagai ground truth maupun pengganti diam-diam YOLO26n-seg.
        Hanya dipanggil secara eksplisit untuk:
          (a) Pembuatan draft poligon awal yang wajib dikoreksi manual di Roboflow/CVAT, atau
          (b) Eksperimen perbandingan baseline metode konvensional vs YOLO26n-seg.
        """
        h, w = img_bgr.shape[:2]
        blurred = cv2.GaussianBlur(img_bgr, (5, 5), 0)
        hsv = cv2.cvtColor(blurred, cv2.COLOR_BGR2HSV)
        mask_hsv = cv2.inRange(hsv, np.array([18, 25, 25]), np.array([98, 255, 255]))

        b, g, r = cv2.split(blurred.astype(np.float32))
        exg = 2.0 * g - r - b
        mask_exg = np.where(exg > 18.0, 255, 0).astype(np.uint8)

        combined = cv2.bitwise_and(mask_hsv, mask_exg)
        k_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
        k_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
        combined = cv2.morphologyEx(combined, cv2.MORPH_CLOSE, k_close, iterations=2)
        combined = cv2.morphologyEx(combined, cv2.MORPH_OPEN, k_open, iterations=1)

        cnts, _ = cv2.findContours(combined, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        min_area = h * w * min_area_ratio
        draft_leaves = []
        for c in sorted(cnts, key=cv2.contourArea, reverse=True):
            area = cv2.contourArea(c)
            if area < min_area:
                continue
            m = np.zeros((h, w), dtype=np.uint8)
            cv2.drawContours(m, [c], -1, 1, thickness=-1)
            x, y, bw, bh = cv2.boundingRect(c)
            draft_leaves.append({
                "leaf_id": len(draft_leaves) + 1,
                "bbox": [int(x), int(y), int(x + bw), int(y + bh)],
                "conf": 0.50,
                "mask": m,
                "contour": c,
                "area_px": int(area),
                "source": "HSV_DRAFT_COMPARISON_ONLY",
            })
            if len(draft_leaves) >= 25:
                break
        return draft_leaves

    def detect_leaves(
        self,
        img_bgr: np.ndarray,
        conf: float = 0.25,
        iou: float = 0.50,
        imgsz: int = 768,
        allow_hsv_fallback: bool = False,
    ) -> list:
        """
        Fungsi 1 (Metode Utama): Mendeteksi setiap daun menggunakan YOLO26n-seg yang telah dilatih pada daun teh.
        Secara default allow_hsv_fallback=False agar hasil evaluasi dan inferensi
        murni mencerminkan kinerja YOLO26n-seg tanpa tercampur heuristik HSV.
        """
        if self.leaf_seg_model is None and os.path.exists(LEAF_SEG_WEIGHTS):
            self.load_trained_leaf_seg(LEAF_SEG_WEIGHTS)
        if self.leaf_seg_model is None:
            raise RuntimeError(
                "[URUTAN PIPELINE] Model YOLO26n-seg khusus daun teh belum dilatih! "
                "Jalankan Cell 28 (Dataset Segmentasi) dan Cell 30 (Training YOLO26n-seg) terlebih dahulu "
                "sebelum menjalankan deteksi daun terintegrasi di Cell 18, 32, dan 34."
            )

        h, w = img_bgr.shape[:2]
        leaves = []
        res = self.leaf_seg_model.predict(img_bgr, imgsz=imgsz, conf=conf, iou=iou, retina_masks=True, verbose=False)[0]
        if res.masks is not None and len(res.masks) > 0:
            masks_data = res.masks.data.cpu().numpy()
            boxes_data = res.boxes
            raw_candidates = []
            for i in range(len(masks_data)):
                m_bin = (masks_data[i] > 0.5).astype(np.uint8)
                if m_bin.shape[0] != h or m_bin.shape[1] != w:
                    m_bin = cv2.resize(m_bin, (w, h), interpolation=cv2.INTER_NEAREST)
                area_px = int(m_bin.sum())
                if area_px < int(0.003 * h * w):
                    continue
                cnts, _ = cv2.findContours(m_bin, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                if not cnts:
                    continue
                c_main = max(cnts, key=cv2.contourArea)
                x1, y1, x2, y2 = map(int, boxes_data.xyxy[i].tolist())
                cf = float(boxes_data.conf[i].item())
                raw_candidates.append({
                    "bbox": [x1, y1, x2, y2],
                    "conf": cf,
                    "mask": m_bin,
                    "contour": c_main,
                    "area_px": area_px,
                    "source": "YOLO26n-seg",
                })
            raw_candidates.sort(key=lambda d: (d["bbox"][1] // max(1, h // 4), d["bbox"][0]))
            for idx_l, item in enumerate(raw_candidates, 1):
                item["leaf_id"] = idx_l
                leaves.append(item)

        if len(leaves) == 0 and allow_hsv_fallback:
            leaves = self.generate_hsv_draft_or_comparison_masks(img_bgr)
        return leaves

    def detect_lesions(
        self,
        img_bgr: np.ndarray,
        conf: float = 0.25,
        iou: float = 0.45,
        imgsz: int = 768,
        use_tta: bool = True,
        exclude_healthy_leaf_class: bool = True,
    ) -> list:
        """
        REVISI POIN A & POIN D:
        Fungsi 2 (Independen dari Segmentasi Daun): Mendeteksi seluruh bounding box bercak penyakit
        menggunakan YOLO26n+CA (Kelas 0..5, TERMASUK Kelas 3 = 'Helopeltis').
        Jika exclude_healthy_leaf_class=True (default), Kelas 6 ('Healty Leaf' / HEALTHY_LEAF_CLASS_ID=6)
        disaring karena merepresentasikan kotak daun utuh, bukan bercak penyakit.
        """
        h, w = img_bgr.shape[:2]
        if not use_tta:
            res = self.lesion_model.predict(img_bgr, imgsz=imgsz, conf=conf, iou=iou, verbose=False)[0]
            lesions = []
            if res.boxes is not None and len(res.boxes) > 0:
                for b in res.boxes:
                    cid = int(b.cls.item())
                    if exclude_healthy_leaf_class and cid == HEALTHY_LEAF_CLASS_ID:
                        continue
                    x1, y1, x2, y2 = map(int, b.xyxy[0].tolist())
                    lesions.append({
                        "lesion_id": len(lesions) + 1,
                        "cls_id": cid,
                        "cls_name": CLASS_NAMES[cid],
                        "conf": float(b.conf.item()),
                        "bbox": [x1, y1, x2, y2],
                    })
            return lesions

        views = [
            ("orig", img_bgr),
            ("hflip", cv2.flip(img_bgr, 1)),
            ("vflip", cv2.flip(img_bgr, 0)),
            ("rot180", cv2.rotate(img_bgr, cv2.ROTATE_180)),
        ]
        all_boxes, all_scores, all_cls = [], [], []
        for vname, vimg in views:
            r = self.lesion_model.predict(vimg, imgsz=imgsz, conf=conf, iou=iou, verbose=False)[0]
            if r.boxes is None or len(r.boxes) == 0:
                continue
            for b in r.boxes:
                cid = int(b.cls.item())
                if exclude_healthy_leaf_class and cid == HEALTHY_LEAF_CLASS_ID:
                    continue
                cf = float(b.conf.item())
                x1, y1, x2, y2 = b.xyxy[0].cpu().numpy().tolist()
                if vname == "hflip":
                    x1, x2 = w - x2, w - x1
                elif vname == "vflip":
                    y1, y2 = h - y2, h - y1
                elif vname == "rot180":
                    x1, x2 = w - x2, w - x1
                    y1, y2 = h - y2, h - y1
                all_boxes.append([max(0, x1), max(0, y1), min(w, x2), min(h, y2)])
                all_scores.append(cf)
                all_cls.append(cid)

        if not all_boxes:
            return []

        boxes_t = torch.tensor(all_boxes, dtype=torch.float32)
        scores_t = torch.tensor(all_scores, dtype=torch.float32)
        cls_t = torch.tensor(all_cls, dtype=torch.int64)
        offsets = cls_t.to(torch.float32) * (max(w, h) + 4096.0)
        keep = torch_nms(boxes_t + offsets.unsqueeze(1), scores_t, iou_threshold=iou)

        lesions = []
        for j, k_idx in enumerate(keep.tolist(), 1):
            bx = [int(round(v)) for v in all_boxes[k_idx]]
            cid = int(all_cls[k_idx])
            lesions.append({
                "lesion_id": j,
                "cls_id": cid,
                "cls_name": CLASS_NAMES[cid],
                "conf": float(all_scores[k_idx]),
                "bbox": bx,
            })
        return lesions

    @staticmethod
    def associate_leaves_and_lesions(
        leaves: list,
        lesions: list,
        min_containment: float = 0.30,
        ambiguity_margin: float = 0.20,
        boundary_band_px: float = 6.0,
    ) -> dict:
        """
        REVISI CELL 16 & CELL 32 (Prioritas 2):
        1. Menghitung Lesion Containment Ratio (LCR) antara setiap bercak B_j dan setiap mask daun M_i:
               LCR(B_j, M_i) = |B_j ∩ M_i| / |B_j|
        2. Penanganan eksplisit bercak di antara dua daun / overlap ambigu (AMBIGUOUS_OVERLAP):
           - Jika terdapat >= 2 daun kandidat dengan LCR >= 0.25, dan selisih antara dua daun teratas
             (LCR_top1 - LCR_top2) < ambiguity_margin (default 0.20), ATAU titik pusat bercak berada
             di perbatasan kedua daun (|dist_contour| <= boundary_band_px dengan LCR keduanya >= 0.25),
             maka bercak ditandai sebagai 'AMBIGUOUS_OVERLAP' (tidak dipaksakan ke salah satu daun).
        3. Status daun tanpa bercak diubah dari 'Healthy' otomatis menjadi 'No lesion detected'.
        """
        leaf_map = {}
        for lf in leaves:
            lf_copy = dict(lf)
            lf_copy["lesions"] = []
            lf_copy["ambiguous_lesions"] = []
            leaf_map[lf["leaf_id"]] = lf_copy

        unassociated_lesions = []
        ambiguous_lesions = []

        for les in lesions:
            x1, y1, x2, y2 = les["bbox"]
            box_area = max(1, (x2 - x1) * (y2 - y1))
            cx, cy = (x1 + x2) / 2.0, (y1 + y2) / 2.0

            candidates = []
            for lf_id, lf in leaf_map.items():
                m_crop = lf["mask"][max(0, y1):max(0, y2), max(0, x1):max(0, x2)]
                inter_px = int(m_crop.sum()) if m_crop.size > 0 else 0
                lcr = inter_px / float(box_area)
                dist_c = float(cv2.pointPolygonTest(lf["contour"], (float(cx), float(cy)), True))
                if lcr > 0.0 or dist_c >= -boundary_band_px:
                    candidates.append({
                        "leaf_id": lf_id,
                        "lcr": float(lcr),
                        "dist_contour": dist_c,
                    })

            candidates.sort(key=lambda c: (c["lcr"], c["dist_contour"]), reverse=True)
            les_copy = dict(les)

            if not candidates or (candidates[0]["lcr"] < min_containment and candidates[0]["dist_contour"] < 0.0):
                les_copy["leaf_id"] = None
                les_copy["containment_ratio"] = round(candidates[0]["lcr"], 4) if candidates else 0.0
                les_copy["association_status"] = "UNASSOCIATED_BACKGROUND"
                les_copy["candidate_leaf_ids"] = []
                unassociated_lesions.append(les_copy)
                continue

            top1 = candidates[0]
            is_ambiguous = False
            if len(candidates) >= 2:
                top2 = candidates[1]
                lcr_gap = top1["lcr"] - top2["lcr"]
                both_substantial = (top1["lcr"] >= 0.25 and top2["lcr"] >= 0.25)
                near_both_borders = (
                    abs(top1["dist_contour"]) <= boundary_band_px
                    and abs(top2["dist_contour"]) <= boundary_band_px
                    and top2["lcr"] >= 0.20
                )
                if (both_substantial and lcr_gap < ambiguity_margin) or near_both_borders:
                    is_ambiguous = True
                    les_copy["leaf_id"] = None
                    les_copy["containment_ratio"] = round(top1["lcr"], 4)
                    les_copy["second_containment_ratio"] = round(top2["lcr"], 4)
                    les_copy["lcr_margin"] = round(lcr_gap, 4)
                    les_copy["association_status"] = "AMBIGUOUS_OVERLAP"
                    les_copy["candidate_leaf_ids"] = [top1["leaf_id"], top2["leaf_id"]]
                    ambiguous_lesions.append(les_copy)
                    leaf_map[top1["leaf_id"]]["ambiguous_lesions"].append(les_copy)
                    leaf_map[top2["leaf_id"]]["ambiguous_lesions"].append(les_copy)

            if not is_ambiguous:
                best_id = top1["leaf_id"]
                les_copy["leaf_id"] = best_id
                les_copy["containment_ratio"] = round(top1["lcr"], 4)
                les_copy["second_containment_ratio"] = round(candidates[1]["lcr"], 4) if len(candidates) >= 2 else 0.0
                les_copy["lcr_margin"] = round(top1["lcr"] - (candidates[1]["lcr"] if len(candidates) >= 2 else 0.0), 4)
                les_copy["association_status"] = "ASSOCIATED"
                les_copy["candidate_leaf_ids"] = [best_id]
                leaf_map[best_id]["lesions"].append(les_copy)

        # Ringkas status tiap daun (Daun tanpa bercak diberi status 'No lesion detected', BUKAN otomatis 'Healthy')
        for lf_id, lf in leaf_map.items():
            les_list = lf["lesions"]
            amb_list = lf["ambiguous_lesions"]
            if len(les_list) == 0:
                lf["status"] = "No lesion detected" if len(amb_list) == 0 else "No confirmed lesion (Ambiguous border overlap)"
                lf["dominant_cls_id"] = None
                lf["dominant_cls_name"] = "No lesion detected"
                lf["max_conf"] = float(lf["conf"])
                lf["severity_pct"] = 0.0
            else:
                best_les = max(les_list, key=lambda x: x["conf"])
                lf["status"] = "Diseased"
                lf["dominant_cls_id"] = best_les["cls_id"]
                lf["dominant_cls_name"] = best_les["cls_name"]
                lf["max_conf"] = float(best_les["conf"])
                lesion_mask = np.zeros_like(lf["mask"], dtype=np.uint8)
                for l_item in les_list:
                    bx1, by1, bx2, by2 = l_item["bbox"]
                    lesion_mask[max(0, by1):max(0, by2), max(0, bx1):max(0, bx2)] = 1
                infected_px = int(np.logical_and(lf["mask"] > 0, lesion_mask > 0).sum())
                lf["severity_pct"] = round(100.0 * infected_px / max(1, lf["area_px"]), 2)

        return {
            "leaves": list(leaf_map.values()),
            "lesions": lesions,
            "ambiguous_lesions": ambiguous_lesions,
            "unassociated_lesions": unassociated_lesions,
        }

    def analyze_field_image(self, img_bgr: np.ndarray, leaf_conf: float = 0.25, lesion_conf: float = 0.25, use_tta: bool = True, allow_hsv_fallback: bool = False) -> dict:
        leaves = self.detect_leaves(img_bgr, conf=leaf_conf, allow_hsv_fallback=allow_hsv_fallback)
        lesions = self.detect_lesions(img_bgr, conf=lesion_conf, use_tta=use_tta, exclude_healthy_leaf_class=True)
        return self.associate_leaves_and_lesions(leaves, lesions)

    def detect_and_visualize(
        self,
        img_bgr: np.ndarray,
        leaf_conf: float = 0.25,
        lesion_conf: float = 0.25,
        use_tta: bool = True,
        allow_hsv_draft_if_untrained: bool = True,
    ) -> dict:
        """
        Fungsi pengujian interaktif untuk menampilkan deteksi daun & seluruh bercak penyakit
        (kompatibel dengan pemanggilan `detector.detect_and_visualize(img_bgr)` pada unggahan file manual).
        - Jika `YOLO26n-seg` (`self.leaf_seg_model`) sudah dilatih di Cell 30, menggunakan `YOLO26n-seg`.
        - Jika belum dilatih dan `allow_hsv_draft_if_untrained=True`, menggunakan kontur pembanding HSV
          serta tetap menampilkan 100% bounding box bercak yang terdeteksi oleh `YOLO26n + CA`.
        """
        use_hsv_fallback = (self.leaf_seg_model is None) and allow_hsv_draft_if_untrained
        analysis = self.analyze_field_image(
            img_bgr,
            leaf_conf=leaf_conf,
            lesion_conf=lesion_conf,
            use_tta=use_tta,
            allow_hsv_fallback=use_hsv_fallback,
        )

        canvas = img_bgr.copy()
        overlay = img_bgr.copy()
        leaf_colors = [
            (60, 200, 60), (220, 180, 40), (200, 100, 220),
            (40, 200, 220), (255, 140, 60), (140, 220, 100),
        ]

        for lf in analysis["leaves"]:
            lid = lf["leaf_id"]
            col_leaf = leaf_colors[(lid - 1) % len(leaf_colors)]
            overlay[lf["mask"] > 0] = col_leaf
            cv2.drawContours(canvas, [lf["contour"]], -1, col_leaf, 2, cv2.LINE_AA)

        canvas = cv2.addWeighted(overlay, 0.22, canvas, 0.78, 0)

        for lf in analysis["leaves"]:
            lid = lf["leaf_id"]
            lx1, ly1, lx2, ly2 = lf["bbox"]
            header = f"Leaf #{lid}: {lf['dominant_cls_name']} ({len(lf['lesions'])} lesion, sev {lf['severity_pct']:.1f}%)"
            (tw, th), _ = cv2.getTextSize(header, cv2.FONT_HERSHEY_SIMPLEX, 0.48, 2)
            hy = max(ly1 - 6, th + 6)
            bg_col = (30, 130, 30) if lf["status"].startswith("No lesion") else (0, 70, 200)
            cv2.rectangle(canvas, (lx1, hy - th - 6), (lx1 + tw + 8, hy + 2), bg_col, -1)
            cv2.putText(canvas, header, (lx1 + 4, hy - 2), cv2.FONT_HERSHEY_SIMPLEX, 0.48, (255, 255, 255), 1, cv2.LINE_AA)

            for les in lf["lesions"]:
                bx1, by1, bx2, by2 = les["bbox"]
                cid = les["cls_id"]
                cf = les["conf"]
                bcol = PALETTE_BGR[cid % len(PALETTE_BGR)]
                cv2.rectangle(canvas, (bx1, by1), (bx2, by2), bcol, 2)
                ltag = f"L#{lid} {CLASS_NAMES[cid][:10]} {cf:.2f}"
                (ltw, lth), _ = cv2.getTextSize(ltag, cv2.FONT_HERSHEY_SIMPLEX, 0.40, 1)
                lty = max(by1 - 3, lth + 4)
                cv2.rectangle(canvas, (bx1, lty - lth - 4), (bx1 + ltw + 4, lty + 2), bcol, -1)
                cv2.putText(canvas, ltag, (bx1 + 2, lty - 2), cv2.FONT_HERSHEY_SIMPLEX, 0.40, (255, 255, 255), 1, cv2.LINE_AA)

        for amb in analysis.get("ambiguous_lesions", []):
            bx1, by1, bx2, by2 = amb["bbox"]
            cand_str = ",".join(f"L#{c}" for c in amb.get("candidate_leaf_ids", []))
            cv2.rectangle(canvas, (bx1, by1), (bx2, by2), (0, 215, 255), 2)
            atag = f"AMBIGUOUS({cand_str}) {amb['cls_name'][:10]} {amb['conf']:.2f}"
            cv2.putText(canvas, atag, (bx1, max(15, by1 - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.40, (0, 215, 255), 1, cv2.LINE_AA)

        for unassoc in analysis.get("unassociated_lesions", []):
            bx1, by1, bx2, by2 = unassoc["bbox"]
            cid = unassoc["cls_id"]
            cf = unassoc["conf"]
            bcol = PALETTE_BGR[cid % len(PALETTE_BGR)]
            cv2.rectangle(canvas, (bx1, by1), (bx2, by2), bcol, 2)
            utag = f"Lesion#{unassoc['lesion_id']} {CLASS_NAMES[cid][:10]} {cf:.2f}"
            (utw, uth), _ = cv2.getTextSize(utag, cv2.FONT_HERSHEY_SIMPLEX, 0.40, 1)
            uty = max(by1 - 3, uth + 4)
            cv2.rectangle(canvas, (bx1, uty - uth - 4), (bx1 + utw + 4, uty + 2), bcol, -1)
            cv2.putText(canvas, utag, (bx1 + 2, uty - 2), cv2.FONT_HERSHEY_SIMPLEX, 0.40, (255, 255, 255), 1, cv2.LINE_AA)

        seg_mode_label = "YOLO26n-seg" if self.leaf_seg_model is not None else "HSV Draft/Comparison (YOLO26n-seg belum dilatih)"
        fig, axes = plt.subplots(1, 2, figsize=(15, 6.5))
        axes[0].imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
        axes[0].set_title("Citra Lapangan Asli", fontweight="bold")
        axes[0].axis("off")
        axes[1].imshow(cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB))
        axes[1].set_title(
            f"Hasil Deteksi ({seg_mode_label})\n"
            f"Daun: {len(analysis['leaves'])} | Total Bercak: {len(analysis['lesions'])} | Ambigu: {len(analysis['ambiguous_lesions'])}",
            fontweight="bold",
        )
        axes[1].axis("off")
        plt.tight_layout()
        plt.show()

        print(f"  -> Ringkasan: {len(analysis['leaves'])} daun terdeteksi | {len(analysis['lesions'])} bercak penyakit terdeteksi")
        for les in analysis["lesions"]:
            print(f"     * Lesion #{les['lesion_id']}: {les['cls_name']} (conf={les['conf']:.3f}, bbox={les['bbox']})")
        return analysis

field_detector = TeaLeafFieldDetector(lesion_weights=BEST_WEIGHTS, leaf_seg_weights=LEAF_SEG_WEIGHTS)


### [Cell 18 | Dipertahankan & Disesuaikan Urutan Eksekusi (Poin D)] Pengujian Gambar Lapangan (Wajib Dijalankan Setelah Training `YOLO26n-seg` di Cell 28 & 30)

In [ ]:
import pandas as pd
from IPython.display import display

def visualize_field_analysis(img_bgr: np.ndarray, analysis: dict) -> np.ndarray:
    canvas = img_bgr.copy()
    overlay = img_bgr.copy()

    LEAF_COLORS = [
        (60, 200, 60), (220, 180, 40), (200, 100, 220),
        (40, 200, 220), (255, 140, 60), (140, 220, 100)
    ]

    # 1. Gambar mask dan identitas setiap daun
    for lf in analysis["leaves"]:
        lid = lf["leaf_id"]
        col_leaf = LEAF_COLORS[(lid - 1) % len(LEAF_COLORS)]
        overlay[lf["mask"] > 0] = col_leaf
        cv2.drawContours(canvas, [lf["contour"]], -1, col_leaf, 2, cv2.LINE_AA)

    canvas = cv2.addWeighted(overlay, 0.22, canvas, 0.78, 0)

    # 2. Gambar seluruh bounding box bercak yang terasosiasi ke daun
    for lf in analysis["leaves"]:
        lid = lf["leaf_id"]
        lx1, ly1, lx2, ly2 = lf["bbox"]
        header = f"Leaf #{lid}: {lf['dominant_cls_name']} ({len(lf['lesions'])} lesion, sev {lf['severity_pct']:.1f}%)"
        (tw, th), _ = cv2.getTextSize(header, cv2.FONT_HERSHEY_SIMPLEX, 0.48, 2)
        hy = max(ly1 - 6, th + 6)
        bg_col = (30, 130, 30) if lf["status"].startswith("No lesion") else (0, 70, 200)
        cv2.rectangle(canvas, (lx1, hy - th - 6), (lx1 + tw + 8, hy + 2), bg_col, -1)
        cv2.putText(canvas, header, (lx1 + 4, hy - 2), cv2.FONT_HERSHEY_SIMPLEX, 0.48, (255, 255, 255), 1, cv2.LINE_AA)

        for les in lf["lesions"]:
            bx1, by1, bx2, by2 = les["bbox"]
            cid = les["cls_id"]
            cf = les["conf"]
            bcol = PALETTE_BGR[cid % len(PALETTE_BGR)]
            cv2.rectangle(canvas, (bx1, by1), (bx2, by2), bcol, 2)
            ltag = f"L#{lid} {CLASS_NAMES[cid][:10]} {cf:.2f}"
            (ltw, lth), _ = cv2.getTextSize(ltag, cv2.FONT_HERSHEY_SIMPLEX, 0.40, 1)
            lty = max(by1 - 3, lth + 4)
            cv2.rectangle(canvas, (bx1, lty - lth - 4), (bx1 + ltw + 4, lty + 2), bcol, -1)
            cv2.putText(canvas, ltag, (bx1 + 2, lty - 2), cv2.FONT_HERSHEY_SIMPLEX, 0.40, (255, 255, 255), 1, cv2.LINE_AA)

    # 3. Gambar bercak yang memiliki overlap ambigu di antara dua daun (warna kuning-oranye)
    for amb in analysis.get("ambiguous_lesions", []):
        bx1, by1, bx2, by2 = amb["bbox"]
        cand_str = ",".join(f"L#{c}" for c in amb.get("candidate_leaf_ids", []))
        cv2.rectangle(canvas, (bx1, by1), (bx2, by2), (0, 215, 255), 2)
        atag = f"AMBIGUOUS({cand_str}) {amb['cls_name'][:10]} {amb['conf']:.2f}"
        cv2.putText(canvas, atag, (bx1, max(15, by1 - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.40, (0, 215, 255), 1, cv2.LINE_AA)

    return canvas

def visualize_independent_lesions_only(img_bgr: np.ndarray, lesions: list) -> np.ndarray:
    """Visualisasi deteksi bercak independen (tanpa segmentasi daun)."""
    canvas = img_bgr.copy()
    for les in lesions:
        bx1, by1, bx2, by2 = les["bbox"]
        cid = les["cls_id"]
        cf = les["conf"]
        bcol = PALETTE_BGR[cid % len(PALETTE_BGR)]
        cv2.rectangle(canvas, (bx1, by1), (bx2, by2), bcol, 2)
        ltag = f"#{les['lesion_id']} {CLASS_NAMES[cid][:12]} {cf:.2f}"
        (ltw, lth), _ = cv2.getTextSize(ltag, cv2.FONT_HERSHEY_SIMPLEX, 0.42, 1)
        lty = max(by1 - 3, lth + 4)
        cv2.rectangle(canvas, (bx1, lty - lth - 4), (bx1 + ltw + 4, lty + 2), bcol, -1)
        cv2.putText(canvas, ltag, (bx1 + 2, lty - 2), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (255, 255, 255), 1, cv2.LINE_AA)
    return canvas

def run_integrated_field_image_testing(detector: TeaLeafFieldDetector, sample_paths: list = None):
    """
    REVISI POIN D (Cell 18):
    Memastikan pengujian sistem terintegrasi (daun + bercak + asosiasi) hanya dijalankan
    setelah model segmentasi khusus daun teh (`LEAF_SEG_WEIGHTS`) selesai dilatih pada Cell 28 & 30.
    """
    if sample_paths is None:
        field_test_dir = os.path.join(FIELD_500_DIR, "images", "test")
        sample_paths = sorted(glob.glob(os.path.join(field_test_dir, "*.*")))[:4]
    if not sample_paths:
        print("[INFO] Tidak ditemukan citra pada split test lapangan.")
        return None

    if detector.leaf_seg_model is None and os.path.exists(LEAF_SEG_WEIGHTS):
        detector.load_trained_leaf_seg(LEAF_SEG_WEIGHTS)

    if detector.leaf_seg_model is None:
        print("=" * 92)
        print("  [PERINGATAN URUTAN EKSEKUSI (POIN D)] — MODEL SEGMENTASI DAUN TEH BELUM DILATIH")
        print("  Pengujian sistem terintegrasi (Daun + Bercak) memerlukan `LEAF_SEG_WEIGHTS` dari Cell 30.")
        print("  Silakan jalankan terlebih dahulu: Cell 22 -> Cell 24 -> Cell 28 -> Cell 30,")
        print("  kemudian jalankan kembali Cell 18, Cell 32, dan Cell 34.")
        print("=" * 92)
        return None

    fig, axes = plt.subplots(2, 2, figsize=(18, 13))
    axes = axes.flatten()
    report_rows = []

    for idx, fpath in enumerate(sample_paths):
        img_bgr = cv2.imread(fpath)
        res_dict = detector.analyze_field_image(img_bgr, leaf_conf=0.25, lesion_conf=0.25, use_tta=True)
        vis_bgr = visualize_field_analysis(img_bgr, res_dict)
        axes[idx].imshow(cv2.cvtColor(vis_bgr, cv2.COLOR_BGR2RGB))
        axes[idx].axis("off")
        axes[idx].set_title(
            f"{os.path.basename(fpath)}\nDaun: {len(res_dict['leaves'])} | Bercak: {len(res_dict['lesions'])} | Ambigu: {len(res_dict['ambiguous_lesions'])}",
            fontsize=10.5,
            fontweight="bold",
        )

        for lf in res_dict["leaves"]:
            if lf["lesions"]:
                for les in lf["lesions"]:
                    report_rows.append({
                        "Gambar": os.path.basename(fpath),
                        "Leaf ID": f"Leaf #{lf['leaf_id']}",
                        "Sumber Mask": lf["source"],
                        "Status Daun": lf["status"],
                        "Kelas Bercak": les["cls_name"],
                        "Confidence Bercak": f"{les['conf']:.3f}",
                        "BBox Bercak": str(les["bbox"]),
                        "Containment Ratio": f"{les['containment_ratio']:.2f}",
                        "Status Asosiasi": les["association_status"],
                        "Severity Daun (%)": f"{lf['severity_pct']:.2f}%",
                    })
            else:
                report_rows.append({
                    "Gambar": os.path.basename(fpath),
                    "Leaf ID": f"Leaf #{lf['leaf_id']}",
                    "Sumber Mask": lf["source"],
                    "Status Daun": lf["status"],
                    "Kelas Bercak": "No lesion detected",
                    "Confidence Bercak": f"{lf['conf']:.3f}",
                    "BBox Bercak": "-",
                    "Containment Ratio": "-",
                    "Status Asosiasi": "NO_LESION_ON_LEAF",
                    "Severity Daun (%)": "0.00%",
                })

        for amb in res_dict.get("ambiguous_lesions", []):
            report_rows.append({
                "Gambar": os.path.basename(fpath),
                "Leaf ID": f"Ambiguous ({amb['candidate_leaf_ids']})",
                "Sumber Mask": "Multi-Leaf Boundary",
                "Status Daun": "Overlap Ambigu",
                "Kelas Bercak": amb["cls_name"],
                "Confidence Bercak": f"{amb['conf']:.3f}",
                "BBox Bercak": str(amb["bbox"]),
                "Containment Ratio": f"{amb['containment_ratio']:.2f} vs {amb['second_containment_ratio']:.2f}",
                "Status Asosiasi": "AMBIGUOUS_OVERLAP",
                "Severity Daun (%)": "-",
            })

    plt.suptitle("Pengujian Sistem Terintegrasi pada Citra Kebun Teh Lapangan (Held-Out Field Test)", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()

    df_field_report = pd.DataFrame(report_rows)
    display(df_field_report)
    return df_field_report

df_field_report = run_integrated_field_image_testing(field_detector)


### [Cell 20 | Dipertahankan] Backup Komprehensif (Checkpoint Model, Konfigurasi YAML, Manifest Split, & Hasil Eksperimen)

In [ ]:
import datetime

def backup_experiment_artifacts(tag_name: str = "baseline_and_configs"):
    ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    gdrive_root = "/content/drive/MyDrive"
    if os.path.exists(gdrive_root):
        dst_root = os.path.join(gdrive_root, "SKRIPSI_TEA_LEAF_BACKUP", f"{tag_name}_{ts}")
    else:
        dst_root = os.path.join(BASE_WORK_DIR, "LOCAL_BACKUP", f"{tag_name}_{ts}")

    os.makedirs(dst_root, exist_ok=True)

    cfg_dst = os.path.join(dst_root, "configs_and_manifests")
    os.makedirs(cfg_dst, exist_ok=True)
    for f_cfg in [MODEL_YAML, SINGLE_LEAF_YAML, FIELD_500_YAML, FIELD_MANIFEST_JSON, REMAPPED_PT]:
        if os.path.exists(f_cfg):
            shutil.copy2(f_cfg, os.path.join(cfg_dst, os.path.basename(f_cfg)))

    runs_dst = os.path.join(dst_root, "phase3_cls_opt_runs")
    if os.path.exists(PROJECT_RUNS):
        shutil.copytree(PROJECT_RUNS, runs_dst, dirs_exist_ok=True)

    zip_archive = shutil.make_archive(dst_root, "zip", dst_root)
    print(f"[BACKUP SUKSES] Seluruh checkpoint, konfigurasi YAML, manifest split, dan metrik disimpan ke:\n  -> Folder : {dst_root}\n  -> Arsip  : {zip_archive}")
    return dst_root

backup_dir_path = backup_experiment_artifacts("phase3_yolo26n_ca_experiment")


### [Cell 22 | Prioritas 2 (Revisi Audit Poin A, D, & E)] Pseudo-Labeling Independen (`detect_lesions`), Pemeriksaan Gambar Tanpa Prediksi, & Persetujuan Eksplisit Anotator

In [ ]:
# ==============================================================================
# REVISI CELL 22 (Prioritas 2 + Perbaikan Audit Poin A, Poin D, & Poin E):
# 1. Poin D (Independensi dari Segmentasi Daun):
#    - Memanggil `detector.detect_lesions()` secara INDEPENDEN (bukan `analyze_field_image()`),
#      sehingga bercak yang berhasil dideteksi oleh YOLO26n+CA tidak pernah hilang
#      meskipun model segmentasi daun belum dilatih atau gagal mengenali daun.
# 2. Poin A (Konsistensi Kelas Penyakit vs Healthy Leaf):
#    - Menyertakan penuh Kelas 3 (`Helopeltis`) sebagai bercak penyakit (`cid in DISEASE_LESION_CLASS_IDS`).
#    - Menyaring hanya Kelas 6 (`HEALTHY_LEAF_CLASS_ID = 6`, `Healty Leaf`).
# 3. Poin E (Jaminan Verifikasi Manual & Pemeriksaan Gambar Tanpa Prediksi):
#    - Gambar dengan 0 prediksi bercak TIDAK dibuang begitu saja, melainkan tetap dimasukkan
#      ke dalam `01_candidates_for_review` (dengan file `.txt` draft kosong) berstatus
#      `priority_band = 'ZERO_PRED_MUST_CHECK_MISSED_LESIONS'`, agar anotator manusia wajib
#      memeriksa apakah ada bercak kecil yang terlewat atau mengonfirmasinya sebagai sampel negatif.
#    - Fungsi `save_and_verify_manually_corrected_annotations()` HANYA memberikan status
#      `manual_verified = True` dan `qa_status = 'MANUALLY_VERIFIED_APPROVED'` apabila
#      terdapat PERSETUJUAN EKSPLISIT ANOTATOR (`annotator_id` terisi dan
#      `review_decision` bernilai `'APPROVED_POSITIVE'` atau `'APPROVED_NEGATIVE_NO_LESION'`).
# ==============================================================================
import csv
import datetime

PSEUDO_ROOT         = "/content/pseudo_labels_lesion_field_train"
PSEUDO_CAND_DIR     = os.path.join(PSEUDO_ROOT, "01_candidates_for_review")
PSEUDO_REJECT_DIR   = os.path.join(PSEUDO_ROOT, "02_rejected_low_quality")
PSEUDO_VERIFIED_DIR = os.path.join(PSEUDO_ROOT, "03_verified_for_finetune")
PSEUDO_VISUAL_DIR   = os.path.join(PSEUDO_ROOT, "04_visual_inspection_sheets")

for d in [PSEUDO_CAND_DIR, PSEUDO_REJECT_DIR, PSEUDO_VERIFIED_DIR, PSEUDO_VISUAL_DIR]:
    os.makedirs(os.path.join(d, "images"), exist_ok=True)
    os.makedirs(os.path.join(d, "labels"), exist_ok=True)

def generate_lesion_pseudo_labels_for_manual_review(
    detector: TeaLeafFieldDetector,
    field_train_img_dir: str,
    conf_high: float = 0.55,
    conf_low: float = 0.25,
):
    """
    Menghasilkan draft pseudo-label bercak penyakit pada `field_train` menggunakan
    `detector.detect_lesions()` secara independen (tanpa bergantung pada segmentasi daun).
    Seluruh gambar (termasuk yang memiliki 0 prediksi bercak) dimasukkan ke dalam
    `01_candidates_for_review` dan `manual_review_checklist.csv` untuk diperiksa manual.
    """
    train_imgs = sorted(glob.glob(os.path.join(field_train_img_dir, "*.*")))
    review_rows = []

    for img_p in train_imgs:
        img_bgr = cv2.imread(img_p)
        if img_bgr is None:
            continue
        h, w = img_bgr.shape[:2]
        bname = os.path.basename(img_p)
        stem = os.path.splitext(bname)[0]

        # REVISI POIN D & POIN A: Gunakan detect_lesions() secara independen; sertakan Kelas 3 (Helopeltis), saring Kelas 6 (Healty Leaf)
        raw_lesions = detector.detect_lesions(
            img_bgr,
            conf=conf_low,
            iou=0.45,
            imgsz=IMGSZ,
            use_tta=True,
            exclude_healthy_leaf_class=True,
        )
        valid_lesions = []
        for les in raw_lesions:
            bx1, by1, bx2, by2 = les["bbox"]
            area_ratio = ((bx2 - bx1) * (by2 - by1)) / float(w * h)
            if 0.0002 <= area_ratio <= 0.45 and les["cls_id"] in DISEASE_LESION_CLASS_IDS:
                valid_lesions.append(les)

        vis_sheet = visualize_independent_lesions_only(img_bgr, valid_lesions)
        cv2.imwrite(os.path.join(PSEUDO_VISUAL_DIR, "images", bname), vis_sheet)

        yolo_lines = []
        for l in valid_lesions:
            bx1, by1, bx2, by2 = l["bbox"]
            cx = ((bx1 + bx2) / 2.0) / w
            cy = ((by1 + by2) / 2.0) / h
            bw = (bx2 - bx1) / float(w)
            bh = (by2 - by1) / float(h)
            yolo_lines.append(f"{l['cls_id']} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")

        # REVISI POIN E: Simpan SEMUA gambar (termasuk 0 prediksi bercak) ke 01_candidates_for_review
        # agar anotator manusia wajib memeriksa gambar tanpa prediksi untuk menemukan bercak yang terlewat!
        shutil.copy2(img_p, os.path.join(PSEUDO_CAND_DIR, "images", bname))
        cand_lbl_path = os.path.join(PSEUDO_CAND_DIR, "labels", stem + ".txt")
        with open(cand_lbl_path, "w", encoding="utf-8") as f:
            f.write(("\n".join(yolo_lines) + "\n") if yolo_lines else "")
        draft_md5 = compute_file_md5(cand_lbl_path)

        if not valid_lesions:
            mean_cf, min_cf = 0.0, 0.0
            priority_band = "ZERO_PRED_MUST_CHECK_MISSED_LESIONS"
            note_str = "0 prediksi otomatis: WAJIB diperiksa manusia untuk mencari bercak terlewat atau set APPROVED_NEGATIVE_NO_LESION"
        else:
            confs = [l["conf"] for l in valid_lesions]
            mean_cf, min_cf = float(np.mean(confs)), float(np.min(confs))
            priority_band = "HIGH_CONF_CANDIDATE" if min_cf >= conf_high else "LOW_CONF_CANDIDATE"
            note_str = "Wajib diperiksa & dikoreksi manual; isi annotator_id & review_decision sebelum verifikasi"

        review_rows.append({
            "image": bname,
            "stem": stem,
            "num_lesions_draft": len(valid_lesions),
            "num_lesions_corrected": 0,
            "mean_conf": round(mean_cf, 4),
            "min_conf": round(min_cf, 4),
            "priority_band": priority_band,
            "draft_label_md5": draft_md5,
            "corrected_label_md5": "",
            "annotator_id": "",
            "review_decision": "UNREVIEWED",
            "human_action": "PENDING",
            "review_timestamp": "",
            "qa_status": "PENDING_MANUAL_REVIEW",
            "manual_verified": False,
            "corrected_label_saved": False,
            "reviewer_notes": note_str,
        })

    csv_path = os.path.join(PSEUDO_ROOT, "manual_review_checklist.csv")
    if review_rows and not os.path.exists(csv_path):
        with open(csv_path, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=list(review_rows[0].keys()))
            writer.writeheader()
            writer.writerows(review_rows)

    df_rev = pd.read_csv(csv_path) if os.path.exists(csv_path) else pd.DataFrame(review_rows)
    print(f"[PSEUDO-LABELING INDEPENDEN SELESAI] Checklist pemeriksaan manual disimpan ke: {csv_path}")
    if not df_rev.empty:
        print(df_rev["priority_band"].value_counts())
    return df_rev

def save_and_verify_manually_corrected_annotations(
    corrected_labels_dir: str,
    review_csv_path: str,
    field_train_img_dir: str,
    verified_dst_dir: str = PSEUDO_VERIFIED_DIR,
    external_signoff_csv: str = None,
) -> pd.DataFrame:
    """
    REVISI POIN A & POIN E (Cell 22):
    1. Memvalidasi berkas `.txt` hasil koreksi manual manusia (mengizinkan Kelas 0..5 termasuk
       Kelas 3 = Helopeltis, dan menolak Kelas 6 = Healty Leaf pada label bercak).
    2. TIDAK memberikan status disetujui hanya karena file `.txt` ditemukan & berformat valid.
       Persetujuan (`manual_verified=True` & `qa_status='MANUALLY_VERIFIED_APPROVED'`) HANYA
       diberikan apabila terdapat persetujuan eksplisit anotator pada `manual_review_checklist.csv`
       (atau `external_signoff_csv`):
         - `annotator_id` wajib terisi (tidak kosong), DAN
         - `review_decision` wajib diset secara eksplisit oleh pemeriksa menjadi:
             * `'APPROVED_POSITIVE'`          (wajib memiliki >= 1 box bercak penyakit valid), atau
             * `'APPROVED_NEGATIVE_NO_LESION'` (wajib memiliki 0 box, mengonfirmasi gambar telah
                                               diperiksa manusia dan benar-benar bebas bercak).
    """
    if not os.path.exists(review_csv_path):
        raise FileNotFoundError(f"Checklist tidak ditemukan: {review_csv_path}")

    df_rev = pd.read_csv(review_csv_path)
    if external_signoff_csv and os.path.exists(external_signoff_csv):
        df_sign = pd.read_csv(external_signoff_csv).set_index("stem")
        for idx, row in df_rev.iterrows():
            st = str(row["stem"])
            if st in df_sign.index:
                for col_s in ("annotator_id", "review_decision", "reviewer_notes"):
                    if col_s in df_sign.columns and pd.notna(df_sign.loc[st, col_s]):
                        df_rev.at[idx, col_s] = str(df_sign.loc[st, col_s])

    if not os.path.exists(corrected_labels_dir):
        print(f"[INFO] Folder hasil koreksi manual '{corrected_labels_dir}' belum tersedia.")
        return df_rev

    verified_count = 0
    skipped_unapproved = 0
    now_ts = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    for idx, row in df_rev.iterrows():
        bname = str(row["image"])
        stem = str(row["stem"])
        annotator = str(row.get("annotator_id", "")).strip()
        if annotator.lower() in ("", "nan", "none"):
            annotator = ""
        decision = str(row.get("review_decision", "UNREVIEWED")).strip().upper()

        # REVISI POIN E: Tolak verifikasi jika belum ada persetujuan eksplisit dari anotator manusia!
        if not annotator or decision not in ("APPROVED_POSITIVE", "APPROVED_NEGATIVE_NO_LESION"):
            if decision == "REJECTED":
                df_rev.at[idx, "qa_status"] = "MANUALLY_REJECTED"
                df_rev.at[idx, "manual_verified"] = False
            else:
                skipped_unapproved += 1
            continue

        corr_lbl_path = os.path.join(corrected_labels_dir, stem + ".txt")
        src_img_path = os.path.join(field_train_img_dir, bname)
        if not (os.path.exists(corr_lbl_path) and os.path.exists(src_img_path)):
            continue

        # Validasi format setiap baris bbox (REVISI POIN A: Kelas 0..5 termasuk Kelas 3 Helopeltis sah; Kelas 6 ditolak)
        clean_lines = []
        is_valid_file = True
        with open(corr_lbl_path, "r", encoding="utf-8") as f:
            for ln in f:
                parts = ln.strip().split()
                if not parts:
                    continue
                if len(parts) != 5:
                    is_valid_file = False
                    break
                cid = int(float(parts[0]))
                cx, cy, bw, bh = map(float, parts[1:5])
                if not (cid in DISEASE_LESION_CLASS_IDS and 0.0 <= cx <= 1.0 and 0.0 <= cy <= 1.0 and 0.0 < bw <= 1.0 and 0.0 < bh <= 1.0):
                    is_valid_file = False
                    break
                clean_lines.append(f"{cid} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")

        if not is_valid_file:
            df_rev.at[idx, "qa_status"] = "INVALID_ANNOTATION_FORMAT"
            df_rev.at[idx, "manual_verified"] = False
            continue

        # Cek konsistensi antara keputusan anotator dan isi file hasil koreksi
        if decision == "APPROVED_POSITIVE" and len(clean_lines) == 0:
            df_rev.at[idx, "qa_status"] = "MISMATCH_DECISION_EMPTY_BOXES"
            df_rev.at[idx, "manual_verified"] = False
            continue
        if decision == "APPROVED_NEGATIVE_NO_LESION" and len(clean_lines) > 0:
            df_rev.at[idx, "qa_status"] = "MISMATCH_DECISION_NONEMPTY_BOXES"
            df_rev.at[idx, "manual_verified"] = False
            continue

        shutil.copy2(src_img_path, os.path.join(verified_dst_dir, "images", bname))
        dst_lbl_path = os.path.join(verified_dst_dir, "labels", stem + ".txt")
        with open(dst_lbl_path, "w", encoding="utf-8") as f:
            f.write(("\n".join(clean_lines) + "\n") if clean_lines else "")

        corr_md5 = compute_file_md5(dst_lbl_path)
        draft_md5 = str(row.get("draft_label_md5", ""))
        n_draft = int(row.get("num_lesions_draft", 0))
        if n_draft == 0 and len(clean_lines) > 0:
            h_action = "ADDED_MISSED_LESIONS"
        elif n_draft == 0 and len(clean_lines) == 0:
            h_action = "CONFIRMED_TRUE_NEGATIVE"
        elif corr_md5 != draft_md5:
            h_action = "EDITED_BOXES"
        else:
            h_action = "CONFIRMED_UNCHANGED"

        df_rev.at[idx, "num_lesions_corrected"] = len(clean_lines)
        df_rev.at[idx, "corrected_label_md5"] = corr_md5
        df_rev.at[idx, "human_action"] = h_action
        df_rev.at[idx, "review_timestamp"] = now_ts
        df_rev.at[idx, "qa_status"] = "MANUALLY_VERIFIED_APPROVED"
        df_rev.at[idx, "manual_verified"] = True
        df_rev.at[idx, "corrected_label_saved"] = True
        df_rev.at[idx, "reviewer_notes"] = f"Signed-off by {annotator} ({decision}, {h_action}: {n_draft} draft -> {len(clean_lines)} boxes)"
        verified_count += 1

    df_rev.to_csv(review_csv_path, index=False)
    print(
        f"[VERIFIKASI MANUAL SELESAI] {verified_count} berkas disetujui anotator & disimpan ke: {verified_dst_dir} | "
        f"{skipped_unapproved} berkas masih menunggu sign-off eksplisit (annotator_id & review_decision)."
    )
    return df_rev

df_pseudo_review = generate_lesion_pseudo_labels_for_manual_review(
    detector=field_detector,
    field_train_img_dir=os.path.join(FIELD_500_DIR, "images", "train"),
    conf_high=0.55,
    conf_low=0.25,
)

MANUAL_CORRECTED_LABELS_INPUT = "/content/manual_corrected_field_train_labels"
ANNOTATOR_SIGNOFF_CSV_INPUT   = "/content/annotator_signoff_manifest.csv"
if os.path.exists(MANUAL_CORRECTED_LABELS_INPUT):
    df_pseudo_review = save_and_verify_manually_corrected_annotations(
        corrected_labels_dir=MANUAL_CORRECTED_LABELS_INPUT,
        review_csv_path=os.path.join(PSEUDO_ROOT, "manual_review_checklist.csv"),
        field_train_img_dir=os.path.join(FIELD_500_DIR, "images", "train"),
        verified_dst_dir=PSEUDO_VERIFIED_DIR,
        external_signoff_csv=ANNOTATOR_SIGNOFF_CSV_INPUT,
    )
display(df_pseudo_review.head(15))


### [Cell 24 | Prioritas 1 — WAJIB (Revisi Audit Poin B, C, E, & F)] Fine-Tuning Menggunakan Data Lapangan Berlabel Terverifikasi (`manual_verified=True`, Sign-Off Anotator, & Impor Roboflow Tanpa Duplikat)

In [ ]:
# ==============================================================================
# REVISI CELL 24 (Prioritas 1 + Perbaikan Audit Poin B, C, E, & F):
# 1. Poin E (Filter Persetujuan Eksplisit Anotator):
#    - HANYA menerima baris dengan `manual_verified == True`,
#      `qa_status == 'MANUALLY_VERIFIED_APPROVED'`, `annotator_id` tidak kosong,
#      dan `review_decision in ('APPROVED_POSITIVE', 'APPROVED_NEGATIVE_NO_LESION')`
#      dari folder `03_verified_for_finetune` (`PSEUDO_VERIFIED_DIR`).
# 2. Poin B & Poin F (Impor Roboflow Tanpa Duplikat Foto & Path Label Akurat):
#    - Menggunakan `find_matching_yolo_label_file()` (mendukung struktur standar
#      Roboflow `train/labels/nama.txt`) dan `match_image_to_field_manifest()`
#      (menggunakan `compute_file_md5()`).
#    - Menyimpan label langsung ke nama kanonik asli (`canonical_dst_stem + '.txt'`)
#      tanpa menyalin ulang foto dengan nama `.rf.<hash>.jpg` sehingga 0 duplikat foto!
# 3. Poin C: Memastikan `import yaml` tersedia sebelum `yaml.dump(...)`.
# ==============================================================================
import yaml

FINETUNE_DATASET_DIR = "/content/tea_field_finetune_dataset"
FINETUNE_YAML        = os.path.join(BASE_WORK_DIR, "field_finetune_data.yaml")
ROBOFLOW_FIELD_VERIFIED_ZIP = "/content/roboflow_lapangan_verified.zip"

def build_leakfree_finetune_dataset() -> tuple:
    if os.path.exists(FINETUNE_DATASET_DIR):
        shutil.rmtree(FINETUNE_DATASET_DIR)
    for sp in ["train", "val", "test"]:
        os.makedirs(os.path.join(FINETUNE_DATASET_DIR, "images", sp), exist_ok=True)
        os.makedirs(os.path.join(FINETUNE_DATASET_DIR, "labels", sp), exist_ok=True)

    with open(FIELD_MANIFEST_JSON, "r", encoding="utf-8") as f:
        field_manifest = json.load(f)

    # 1. REVISI POIN B & POIN F: Jika ada bundel anotasi lapangan manual dari Roboflow,
    #    gunakan find_matching_yolo_label_file() & match_image_to_field_manifest() agar
    #    label masuk ke nama file kanonik asli tanpa membuat foto duplikat!
    if os.path.exists(ROBOFLOW_FIELD_VERIFIED_ZIP):
        tmp_rf = "/content/_tmp_rf_field_verified"
        if os.path.exists(tmp_rf):
            shutil.rmtree(tmp_rf)
        with zipfile.ZipFile(ROBOFLOW_FIELD_VERIFIED_ZIP, "r") as zf:
            zf.extractall(tmp_rf)

        mapped_counts = {"train": 0, "val": 0, "test": 0}
        seen_canonical_stems = set()
        for root, _, files in os.walk(tmp_rf):
            for fn in sorted(files):
                if not fn.lower().endswith((".jpg", ".jpeg", ".png")):
                    continue
                img_p = os.path.join(root, fn)
                lbl_cand = find_matching_yolo_label_file(img_p)
                if lbl_cand is None:
                    continue

                entry = match_image_to_field_manifest(img_p, field_manifest)
                if entry is None:
                    continue
                orig_sp = entry["split"]
                canonical_fn = entry["filename"]
                canonical_dst_stem = entry["dst_stem"]
                if orig_sp not in ("train", "val", "test") or canonical_dst_stem in seen_canonical_stems:
                    continue
                seen_canonical_stems.add(canonical_dst_stem)

                dst_img_existing = os.path.join(FIELD_500_DIR, "images", orig_sp, canonical_fn)
                if not os.path.exists(dst_img_existing):
                    shutil.copy2(img_p, dst_img_existing)
                shutil.copy2(lbl_cand, os.path.join(FIELD_500_DIR, "labels", orig_sp, canonical_dst_stem + ".txt"))
                mapped_counts[orig_sp] += 1
        print(f"[INFO] Anotasi terverifikasi dari {ROBOFLOW_FIELD_VERIFIED_ZIP} dipetakan ke nama kanonik split asli (0 duplikat): {mapped_counts}")

    # 2. Salin regularizer Single Leaf ke train split agar tidak terjadi catastrophic forgetting
    sl_imgs = sorted(glob.glob(os.path.join(SINGLE_LEAF_DIR, "images", "train", "*.*")))
    random.seed(42)
    sl_replay = [p for p in sl_imgs if not os.path.basename(p).startswith("bal_")]
    sl_replay = random.sample(sl_replay, min(600, len(sl_replay)))
    for img_p in sl_replay:
        bname = os.path.basename(img_p)
        stem = os.path.splitext(bname)[0]
        lbl_p = os.path.join(SINGLE_LEAF_DIR, "labels", "train", stem + ".txt")
        if os.path.exists(lbl_p):
            shutil.copy2(img_p, os.path.join(FINETUNE_DATASET_DIR, "images", "train", "sl_" + bname))
            shutil.copy2(lbl_p, os.path.join(FINETUNE_DATASET_DIR, "labels", "train", "sl_" + stem + ".txt"))

    # 3. Tambahkan data lapangan train dari FIELD_500_DIR/labels/train (jika ada GT manual langsung)
    verified_field_train_count = 0
    added_train_stems = set()
    for lbl_p in sorted(glob.glob(os.path.join(FIELD_500_DIR, "labels", "train", "*.txt"))):
        stem = os.path.splitext(os.path.basename(lbl_p))[0]
        for ext in (".jpg", ".jpeg", ".png", ".JPG", ".PNG"):
            ip = os.path.join(FIELD_500_DIR, "images", "train", stem + ext)
            if os.path.exists(ip):
                shutil.copy2(ip, os.path.join(FINETUNE_DATASET_DIR, "images", "train", "fld_" + os.path.basename(ip)))
                shutil.copy2(lbl_p, os.path.join(FINETUNE_DATASET_DIR, "labels", "train", "fld_" + stem + ".txt"))
                added_train_stems.add(stem)
                verified_field_train_count += 1
                break

    # 4. FILTER KETAT PSEUDO-LABEL + SIGN-OFF EKSPLISIT ANOTATOR (REVISI CELL 24 — POIN E):
    #    HANYA gunakan baris dengan:
    #      - manual_verified == True
    #      - qa_status == 'MANUALLY_VERIFIED_APPROVED'
    #      - annotator_id tidak kosong
    #      - review_decision in ('APPROVED_POSITIVE', 'APPROVED_NEGATIVE_NO_LESION')
    #    serta file label dari PSEUDO_VERIFIED_DIR (03_verified_for_finetune).
    checklist_csv = os.path.join(PSEUDO_ROOT, "manual_review_checklist.csv")
    approved_pseudo_count = 0
    if os.path.exists(checklist_csv):
        df_rev = pd.read_csv(checklist_csv)
        req_cols = {"manual_verified", "qa_status", "annotator_id", "review_decision"}
        if req_cols.issubset(set(df_rev.columns)):
            is_manual_true = df_rev["manual_verified"].astype(str).str.strip().str.lower().isin(["true", "1"])
            is_status_approved = (df_rev["qa_status"].astype(str).str.strip() == "MANUALLY_VERIFIED_APPROVED")
            has_annotator = ~df_rev["annotator_id"].astype(str).str.strip().str.lower().isin(["", "nan", "none"])
            valid_decision = df_rev["review_decision"].astype(str).str.strip().str.upper().isin(["APPROVED_POSITIVE", "APPROVED_NEGATIVE_NO_LESION"])
            df_approved = df_rev[is_manual_true & is_status_approved & has_annotator & valid_decision]
            for _, r in df_approved.iterrows():
                bname = str(r["image"])
                stem = str(r["stem"])
                if stem in added_train_stems:
                    continue
                v_img = os.path.join(PSEUDO_VERIFIED_DIR, "images", bname)
                v_lbl = os.path.join(PSEUDO_VERIFIED_DIR, "labels", stem + ".txt")
                if os.path.exists(v_img) and os.path.exists(v_lbl):
                    shutil.copy2(v_img, os.path.join(FINETUNE_DATASET_DIR, "images", "train", "fld_" + bname))
                    shutil.copy2(v_lbl, os.path.join(FINETUNE_DATASET_DIR, "labels", "train", "fld_" + stem + ".txt"))
                    added_train_stems.add(stem)
                    approved_pseudo_count += 1

    total_field_train = verified_field_train_count + approved_pseudo_count

    # 5. Salin Ground Truth manual untuk VAL dan TEST (termasuk file .txt kosong untuk gambar tanpa penyakit)
    val_gt_count, test_gt_count = 0, 0
    for sp in ["val", "test"]:
        for img_p in sorted(glob.glob(os.path.join(FIELD_500_DIR, "images", sp, "*.*"))):
            bname = os.path.basename(img_p)
            stem = os.path.splitext(bname)[0]
            lbl_p = os.path.join(FIELD_500_DIR, "labels", sp, stem + ".txt")
            shutil.copy2(img_p, os.path.join(FINETUNE_DATASET_DIR, "images", sp, bname))
            if os.path.exists(lbl_p):
                shutil.copy2(lbl_p, os.path.join(FINETUNE_DATASET_DIR, "labels", sp, stem + ".txt"))
                if sp == "val":
                    val_gt_count += 1
                else:
                    test_gt_count += 1

    with open(FINETUNE_YAML, "w", encoding="utf-8") as f:
        yaml.dump({
            "path": FINETUNE_DATASET_DIR,
            "train": "images/train",
            "val": "images/val",
            "test": "images/test",
            "nc": len(CLASS_NAMES),
            "names": CLASS_NAMES,
        }, f, sort_keys=False)

    print("=" * 92)
    print("  [AUDIT DATASET FINE-TUNING BEBAS KEBOCORAN (CELL 24 — REVISI POIN B, C, E, & F)]")
    print("=" * 92)
    print(f"  - Single-Leaf Replay (Train)                 : {len(sl_replay)} citra")
    print(f"  - Field Train GT Manual Langsung             : {verified_field_train_count} citra")
    print(f"  - Field Train Sign-Off Anotator Terverifikasi: {approved_pseudo_count} citra (annotator_id + review_decision + manual_verified=True)")
    print(f"  - Field Val Ground Truth Manual              : {val_gt_count} citra (0 pseudo-label otomatis)")
    print(f"  - Field Test Ground Truth Manual             : {test_gt_count} citra (0 pseudo-label otomatis)")
    return total_field_train, val_gt_count, test_gt_count

n_ft_train, n_ft_val, n_ft_test = build_leakfree_finetune_dataset()
FT_RUN_NAME = "yolo26n_ca_cls_opt_finetune"
FT_BEST_WEIGHTS = os.path.join(PROJECT_RUNS, FT_RUN_NAME, "weights", "best.pt")

val_completeness = audit_ground_truth_completeness(FINETUNE_DATASET_DIR, split="val")
if n_ft_train > 0 and val_completeness["is_complete"]:
    register_ca_everywhere()
    ft_model = YOLO(BEST_WEIGHTS)
    ft_results = ft_model.train(
        data=FINETUNE_YAML,
        epochs=25,
        imgsz=IMGSZ,
        batch=BATCH,
        device="cuda" if torch.cuda.is_available() else "cpu",
        project=PROJECT_RUNS,
        name=FT_RUN_NAME,
        exist_ok=True,
        lr0=0.0005,
        lrf=0.05,
        cos_lr=True,
        box=8.5,
        cls=1.25,
        dfl=1.5,
        mosaic=0.4,
        close_mosaic=10,
        patience=10,
        seed=42,
        plots=True,
    )
    print(f"[SUKSES] Fine-tuning selesai! Bobot adaptasi lapangan disimpan di: {FT_BEST_WEIGHTS}")
else:
    FT_BEST_WEIGHTS = BEST_WEIGHTS
    print("\n[FINE-TUNING DITANGGUHKAN DEMI INTEGRITAS METODOLOGI]")
    print(f"  - Jumlah data lapangan Train ber-signoff anotator (manual_verified=True): {n_ft_train}")
    print(f"  - Kelengkapan Ground Truth manual split Val: {val_completeness['positive_gt']} positif + {val_completeness['negative_empty_gt']} negatif, {val_completeness['missing_gt']} belum berlabel.")
    print("  -> Pastikan anotator telah mengisi `annotator_id` & `review_decision` pada `manual_review_checklist.csv` lalu jalankan `save_and_verify_manually_corrected_annotations()` di Cell 22.")


### [Cell 26 | Prioritas 3] Evaluasi Fine-Tuning Menggunakan Ground Truth Manual Lengkap (Termasuk Gambar Tanpa Penyakit)

In [ ]:
# ==============================================================================
# REVISI CELL 26 (Prioritas 3):
# 1. Evaluasi WAJIB menggunakan Ground Truth manual yang lengkap pada `field_test`
#    (missing_gt == 0), termasuk menghitung gambar tanpa penyakit yang memiliki
#    berkas `.txt` kosong sebagai Ground Truth negatif yang sah.
# 2. Membandingkan performa Baseline (sebelum adaptasi) vs Fine-Tuned (sesudah
#    adaptasi) secara objektif pada dataset Single Leaf dan dataset Lapangan.
# ==============================================================================
field_test_completeness = audit_ground_truth_completeness(FIELD_500_DIR, split="test")
print(
    f"[AUDIT GT FIELD TEST] Total: {field_test_completeness['total_images']} | "
    f"Positif Berpenyakit: {field_test_completeness['positive_gt']} | "
    f"Negatif Tanpa Penyakit (.txt kosong): {field_test_completeness['negative_empty_gt']} | "
    f"Belum Berlabel (Missing): {field_test_completeness['missing_gt']}"
)

comparison_rows = []

# 1. Perbandingan pada Held-Out Test Set Single Leaf
_, base_sl_m = evaluate_model_on_dataset(BEST_WEIGHTS, SINGLE_LEAF_YAML, SINGLE_LEAF_DIR, "test", "eval_baseline_on_single_leaf", imgsz=IMGSZ)
comparison_rows.append({
    "Dataset Evaluasi (Ground Truth Asli)": "Single Leaf Test Set",
    "Varian Model": "Baseline YOLO26n+CA ( Sebelum Fine-Tuning )",
    "Precision": round(base_sl_m["precision"], 4),
    "Recall": round(base_sl_m["recall"], 4),
    "F1-Score": round(base_sl_m["f1"], 4),
    "mAP@0.50": round(base_sl_m["map50"], 4),
    "mAP@0.50:0.95": round(base_sl_m["map50_95"], 4),
})

if os.path.exists(FT_BEST_WEIGHTS) and FT_BEST_WEIGHTS != BEST_WEIGHTS:
    _, ft_sl_m = evaluate_model_on_dataset(FT_BEST_WEIGHTS, SINGLE_LEAF_YAML, SINGLE_LEAF_DIR, "test", "eval_finetuned_on_single_leaf", imgsz=IMGSZ)
    comparison_rows.append({
        "Dataset Evaluasi (Ground Truth Asli)": "Single Leaf Test Set",
        "Varian Model": "Fine-Tuned YOLO26n+CA ( Sesudah Adaptasi )",
        "Precision": round(ft_sl_m["precision"], 4),
        "Recall": round(ft_sl_m["recall"], 4),
        "F1-Score": round(ft_sl_m["f1"], 4),
        "mAP@0.50": round(ft_sl_m["map50"], 4),
        "mAP@0.50:0.95": round(ft_sl_m["map50_95"], 4),
    })

# 2. Perbandingan pada Held-Out Test Set Lapangan (Wajib 100% Ground Truth Manual Lengkap)
if field_test_completeness["is_complete"]:
    _, base_fld_m = evaluate_model_on_dataset(BEST_WEIGHTS, FIELD_500_YAML, FIELD_500_DIR, "test", "eval_baseline_on_field_gt", imgsz=IMGSZ)
    comparison_rows.append({
        "Dataset Evaluasi (Ground Truth Asli)": f"Field Test Set ({field_test_completeness['positive_gt']} Pos + {field_test_completeness['negative_empty_gt']} Neg)",
        "Varian Model": "Baseline YOLO26n+CA ( Sebelum Fine-Tuning )",
        "Precision": round(base_fld_m["precision"], 4),
        "Recall": round(base_fld_m["recall"], 4),
        "F1-Score": round(base_fld_m["f1"], 4),
        "mAP@0.50": round(base_fld_m["map50"], 4),
        "mAP@0.50:0.95": round(base_fld_m["map50_95"], 4),
    })
    if os.path.exists(FT_BEST_WEIGHTS) and FT_BEST_WEIGHTS != BEST_WEIGHTS:
        _, ft_fld_m = evaluate_model_on_dataset(FT_BEST_WEIGHTS, FIELD_500_YAML, FIELD_500_DIR, "test", "eval_finetuned_on_field_gt", imgsz=IMGSZ)
        comparison_rows.append({
            "Dataset Evaluasi (Ground Truth Asli)": f"Field Test Set ({field_test_completeness['positive_gt']} Pos + {field_test_completeness['negative_empty_gt']} Neg)",
            "Varian Model": "Fine-Tuned YOLO26n+CA ( Sesudah Adaptasi )",
            "Precision": round(ft_fld_m["precision"], 4),
            "Recall": round(ft_fld_m["recall"], 4),
            "F1-Score": round(ft_fld_m["f1"], 4),
            "mAP@0.50": round(ft_fld_m["map50"], 4),
            "mAP@0.50:0.95": round(ft_fld_m["map50_95"], 4),
        })
        comparison_rows.append({
            "Dataset Evaluasi (Ground Truth Asli)": "Field Test Set (Delta Peningkatan)",
            "Varian Model": "Selisih (Fine-Tuned - Baseline)",
            "Precision": round(ft_fld_m["precision"] - base_fld_m["precision"], 4),
            "Recall": round(ft_fld_m["recall"] - base_fld_m["recall"], 4),
            "F1-Score": round(ft_fld_m["f1"] - base_fld_m["f1"], 4),
            "mAP@0.50": round(ft_fld_m["map50"] - base_fld_m["map50"], 4),
            "mAP@0.50:0.95": round(ft_fld_m["map50_95"] - base_fld_m["map50_95"], 4),
        })
else:
    print(
        f"[PERINGATAN] Evaluasi Field Test ditangguhkan karena masih ada {field_test_completeness['missing_gt']} "
        f"citra di `{FIELD_500_DIR}/images/test` yang belum memiliki berkas `.txt` Ground Truth manual."
    )

df_ft_comparison = pd.DataFrame(comparison_rows)
print("\n" + "=" * 95)
print("  [TABEL 26] PERBANDINGAN OBJEKTIF BASELINE VS FINE-TUNED PADA GROUND TRUTH MANUAL LENGKAP")
print("=" * 95)
display(df_ft_comparison)


### [Cell 28 | Prioritas 1 — WAJIB (Cell Baru A)] Persiapan Dataset Segmentasi Daun (Tanpa HSV sebagai Ground Truth, Perbaikan Impor Roboflow, & Verifikasi Manual Per Gambar 100% Cakupan)

In [ ]:
# ==============================================================================
# REVISI CELL 28 / CELL BARU A (Prioritas 1 — WAJIB | Temuan Audit B, C, F, G):
# 1. [Temuan C] Menambahkan `import yaml` secara eksplisit.
# 2. [Temuan B & F] Memperbaiki pembacaan ekspor Roboflow (`roboflow_leaf_seg.zip`):
#    - Menggunakan `find_matching_yolo_label_file(img_p)` untuk struktur standar
#      Roboflow (`train/labels/nama.txt` maupun `labels/train/nama.txt`).
#    - Menggunakan `match_image_to_field_manifest(img_p, field_manifest)` yang
#      memanggil `compute_file_md5(img_p)` dan memetakan setiap foto ke nama
#      kanonik `entry['filename']` (`dst_stem`) sesuai split asli tanpa membuat
#      duplikat foto.
# 3. [Temuan G] Memperketat verifikasi Ground Truth segmentasi:
#    - TIDAK menetapkan `val_verified = (stats['val'] > 0)`.
#    - Menggunakan daftar periksa persetujuan manual per gambar
#      (`leaf_seg_manual_verification_checklist.csv`) yang mencatat:
#      `annotator_id`, `all_target_leaves_annotated` (kelengkapan seluruh
#      instance daun target dalam citra), `includes_hard_or_overlapping_cases`,
#      dan `review_decision == 'APPROVED_COMPLETE_POLYGONS'`.
#    - `val_verified` dan `test_verified` hanya bernilai `True` apabila 100%
#      citra pada split `val` dan `test` di `field_500_split_manifest.json`
#      memiliki poligon valid dan telah disetujui manual (`missing_or_unverified == 0`),
#      sehingga kasus daun sulit/bertumpuk tidak dapat dilewati.
# ==============================================================================
import yaml

LEAF_SEG_DIR               = "/content/tea_leaf_seg_dataset"
LEAF_SEG_YAML              = os.path.join(BASE_WORK_DIR, "leaf_seg_data.yaml")
LEAF_SEG_DRAFT_DIR         = "/content/leaf_seg_drafts_for_manual_annotation"
ROBOFLOW_SEG_ZIP           = "/content/roboflow_leaf_seg.zip"
VERIFIED_SEG_FOLDER        = "/content/leaf_seg_verified_manual"
LEAF_SEG_CHECKLIST_CSV     = os.path.join(LEAF_SEG_DIR, "leaf_seg_manual_verification_checklist.csv")
EXTERNAL_SEG_CHECKLIST_CSV = "/content/leaf_seg_manual_verification_checklist.csv"

def is_polygon_geometrically_feasible(coords_norm: list) -> tuple:
    """
    Memeriksa apakah sebuah poligon daun ternormalisasi [x1, y1, x2, y2, ...] layak:
      - Minimal 3 titik sudut (>= 6 angka) dan berjumlah genap
      - Seluruh koordinat berada pada rentang [0.0, 1.0]
      - Luas poligon (Shoelace formula) berada pada rentang wajar [0.001, 0.96] dari luas gambar
      - Solidity (luas poligon / luas convex hull) >= 0.40
    """
    if len(coords_norm) < 6 or len(coords_norm) % 2 != 0:
        return False, "jumlah_koordinat_tidak_valid"
    pts = np.array(coords_norm, dtype=np.float32).reshape(-1, 2)
    if np.any(pts < 0.0) or np.any(pts > 1.0):
        return False, "koordinat_diluar_0_1"
    pts_scaled = (pts * 1000.0).astype(np.float32)
    area_norm = float(cv2.contourArea(pts_scaled)) / (1000.0 * 1000.0)
    if not (0.001 <= area_norm <= 0.96):
        return False, f"luas_tidak_wajar({area_norm:.5f})"
    hull = cv2.convexHull(pts_scaled)
    hull_area = float(cv2.contourArea(hull)) / (1000.0 * 1000.0)
    solidity = area_norm / max(hull_area, 1e-9)
    if solidity < 0.40:
        return False, f"solidity_rendah({solidity:.2f})"
    return True, "LAYAK"

def export_hsv_draft_polygons_for_annotators_only():
    """
    Mengekspor draft poligon HSV HANYA ke folder `/content/leaf_seg_drafts_for_manual_annotation`
    untuk mempercepat proses menggambar poligon manual di CVAT/Roboflow.
    TIDAK PERNAH dimasukkan ke dalam `LEAF_SEG_DIR` (`tea_leaf_seg_dataset`).
    """
    for sp in ["train", "val", "test"]:
        os.makedirs(os.path.join(LEAF_SEG_DRAFT_DIR, "images", sp), exist_ok=True)
        os.makedirs(os.path.join(LEAF_SEG_DRAFT_DIR, "draft_labels_for_review_only", sp), exist_ok=True)

    draft_count = 0
    for sp in ["train", "val", "test"]:
        for img_p in sorted(glob.glob(os.path.join(FIELD_500_DIR, "images", sp, "*.*"))):
            img_bgr = cv2.imread(img_p)
            if img_bgr is None:
                continue
            h, w = img_bgr.shape[:2]
            bname = os.path.basename(img_p)
            stem = os.path.splitext(bname)[0]
            draft_leaves = TeaLeafFieldDetector.generate_hsv_draft_or_comparison_masks(img_bgr)
            poly_lines = []
            for lf in draft_leaves:
                cnt = lf["contour"]
                peri = cv2.arcLength(cnt, True)
                approx = cv2.approxPolyDP(cnt, 0.004 * peri, True)
                if len(approx) < 4:
                    continue
                pts = approx.reshape(-1, 2)
                coords = []
                for px, py in pts:
                    coords.append(min(1.0, max(0.0, px / float(w))))
                    coords.append(min(1.0, max(0.0, py / float(h))))
                ok, _ = is_polygon_geometrically_feasible(coords)
                if ok:
                    poly_lines.append("0 " + " ".join(f"{c:.6f}" for c in coords))
            if poly_lines:
                shutil.copy2(img_p, os.path.join(LEAF_SEG_DRAFT_DIR, "images", sp, bname))
                with open(os.path.join(LEAF_SEG_DRAFT_DIR, "draft_labels_for_review_only", sp, stem + ".txt"), "w", encoding="utf-8") as f:
                    f.write("\n".join(poly_lines) + "\n")
                draft_count += 1
    print(f"[INFO DRAFT ANOTASI] {draft_count} draft poligon disimpan di `{LEAF_SEG_DRAFT_DIR}` (Bukan Ground Truth, hanya untuk dikoreksi manual).")

def prepare_verified_leaf_segmentation_dataset() -> dict:
    # Cadangkan checklist manual bila sudah ada sebelum membangun ulang folder
    existing_signoff_by_stem = {}
    for cand_csv in [EXTERNAL_SEG_CHECKLIST_CSV, LEAF_SEG_CHECKLIST_CSV]:
        if os.path.exists(cand_csv):
            try:
                df_prev = pd.read_csv(cand_csv).fillna("")
                for _, r in df_prev.iterrows():
                    s_key = str(r.get("dst_stem", "")).strip()
                    if s_key:
                        existing_signoff_by_stem[s_key] = r.to_dict()
            except Exception:
                pass

    if os.path.exists(LEAF_SEG_DIR):
        shutil.rmtree(LEAF_SEG_DIR)
    for sp in ["train", "val", "test"]:
        os.makedirs(os.path.join(LEAF_SEG_DIR, "images", sp), exist_ok=True)
        os.makedirs(os.path.join(LEAF_SEG_DIR, "labels", sp), exist_ok=True)

    with open(FIELD_MANIFEST_JSON, "r", encoding="utf-8") as f:
        field_manifest = json.load(f)

    source_dir = None
    if os.path.exists(ROBOFLOW_SEG_ZIP):
        tmp_seg_rf = "/content/_tmp_rf_leaf_seg"
        if os.path.exists(tmp_seg_rf):
            shutil.rmtree(tmp_seg_rf)
        with zipfile.ZipFile(ROBOFLOW_SEG_ZIP, "r") as zf:
            zf.extractall(tmp_seg_rf)
        source_dir = tmp_seg_rf
        print(f"[INFO] Mengekstrak anotasi poligon daun manual dari `{ROBOFLOW_SEG_ZIP}` dan memetakan ke split asli...")
    elif os.path.exists(VERIFIED_SEG_FOLDER):
        source_dir = VERIFIED_SEG_FOLDER
        print(f"[INFO] Memuat anotasi poligon daun terverifikasi manual dari `{VERIFIED_SEG_FOLDER}`...")

    stats = {
        "train": 0,
        "val": 0,
        "test": 0,
        "rejected_polygons": 0,
        "unmatched_source_files": 0,
        "expected_train_total": len(field_manifest.get("splits", {}).get("train", [])),
        "expected_val_total": len(field_manifest.get("splits", {}).get("val", [])),
        "expected_test_total": len(field_manifest.get("splits", {}).get("test", [])),
        "train_manually_verified_count": 0,
        "val_manually_verified_count": 0,
        "test_manually_verified_count": 0,
        "val_missing_or_unverified_count": 0,
        "test_missing_or_unverified_count": 0,
        "val_verified": False,
        "test_verified": False,
    }

    imported_by_stem = {}
    if source_dir is not None:
        for root, _, files in os.walk(source_dir):
            for fn in sorted(files):
                if not fn.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp")):
                    continue
                img_p = os.path.join(root, fn)
                # [Temuan F] Gunakan pencari pasangan label YOLO/Roboflow yang benar
                lbl_cand = find_matching_yolo_label_file(img_p)
                if lbl_cand is None or not os.path.exists(lbl_cand):
                    continue

                # [Temuan B & F] Cocokkan gambar ke manifest split asli menggunakan compute_file_md5 & stem
                matched_entry = match_image_to_field_manifest(img_p, field_manifest)
                if matched_entry is None:
                    stats["unmatched_source_files"] += 1
                    continue

                orig_sp = matched_entry["split"]
                dst_stem = matched_entry["dst_stem"]
                canonical_fn = matched_entry["filename"]
                if orig_sp not in ("train", "val", "test") or dst_stem in imported_by_stem:
                    continue

                # Validasi kelayakan setiap poligon di dalam file label
                valid_poly_lines = []
                file_rejected_polys = 0
                with open(lbl_cand, "r", encoding="utf-8") as f:
                    for ln in f:
                        parts = ln.strip().split()
                        if not parts:
                            continue
                        if len(parts) < 7 or (len(parts) - 1) % 2 != 0:
                            stats["rejected_polygons"] += 1
                            file_rejected_polys += 1
                            continue
                        coords = [float(x) for x in parts[1:]]
                        ok, _ = is_polygon_geometrically_feasible(coords)
                        if ok:
                            valid_poly_lines.append("0 " + " ".join(f"{c:.6f}" for c in coords))
                        else:
                            stats["rejected_polygons"] += 1
                            file_rejected_polys += 1

                if valid_poly_lines:
                    # Salin dari foto kanonik di FIELD_500_DIR agar tidak ada duplikat nama Roboflow
                    canonical_src = os.path.join(FIELD_500_DIR, "images", orig_sp, canonical_fn)
                    src_to_copy = canonical_src if os.path.exists(canonical_src) else img_p
                    shutil.copy2(src_to_copy, os.path.join(LEAF_SEG_DIR, "images", orig_sp, canonical_fn))
                    with open(os.path.join(LEAF_SEG_DIR, "labels", orig_sp, dst_stem + ".txt"), "w", encoding="utf-8") as f:
                        f.write("\n".join(valid_poly_lines) + "\n")
                    stats[orig_sp] += 1
                    imported_by_stem[dst_stem] = {
                        "num_valid_polygons": len(valid_poly_lines),
                        "rejected_polygons": file_rejected_polys,
                        "label_md5": compute_file_md5(os.path.join(LEAF_SEG_DIR, "labels", orig_sp, dst_stem + ".txt")),
                    }
    else:
        export_hsv_draft_polygons_for_annotators_only()

    # [Temuan G] Bangun & periksa Checklist Verifikasi Manual Per Gambar untuk 100% citra di manifest
    checklist_rows = []
    for sp in ["train", "val", "test"]:
        for entry in field_manifest.get("splits", {}).get(sp, []):
            dst_stem = entry["dst_stem"]
            fn = entry["filename"]
            imp_info = imported_by_stem.get(dst_stem, None)
            has_poly = imp_info is not None and imp_info["num_valid_polygons"] > 0
            n_polys = imp_info["num_valid_polygons"] if imp_info else 0
            n_rej = imp_info["rejected_polygons"] if imp_info else 0

            prev_row = existing_signoff_by_stem.get(dst_stem, {})
            annotator_id = str(prev_row.get("annotator_id", "")).strip()
            all_leaves_done = str(prev_row.get("all_target_leaves_annotated", "False")).strip().lower() in ("true", "1", "yes")
            hard_case_flag = str(prev_row.get("includes_hard_or_overlapping_cases", "True")).strip().lower() in ("true", "1", "yes")
            review_decision = str(prev_row.get("review_decision", "PENDING_MANUAL_POLYGON_REVIEW")).strip()

            # Syarat ketat persetujuan manual per gambar (Temuan G):
            # 1. File poligon ada dan memiliki >= 1 poligon valid tanpa poligon cacat
            # 2. annotator_id wajib terisi
            # 3. all_target_leaves_annotated == True (seluruh daun target sudah diberi poligon)
            # 4. review_decision == 'APPROVED_COMPLETE_POLYGONS'
            is_img_verified = (
                has_poly
                and n_rej == 0
                and bool(annotator_id)
                and all_leaves_done
                and review_decision == "APPROVED_COMPLETE_POLYGONS"
            )
            if is_img_verified:
                stats[f"{sp}_manually_verified_count"] += 1
            else:
                if sp in ("val", "test"):
                    stats[f"{sp}_missing_or_unverified_count"] += 1

            checklist_rows.append({
                "split": sp,
                "filename": fn,
                "dst_stem": dst_stem,
                "group_id": entry.get("group_id", ""),
                "has_valid_polygon_file": has_poly,
                "num_valid_polygons": n_polys,
                "rejected_polygons": n_rej,
                "annotator_id": annotator_id,
                "all_target_leaves_annotated": all_leaves_done,
                "includes_hard_or_overlapping_cases": hard_case_flag,
                "review_decision": review_decision if is_img_verified else "PENDING_MANUAL_POLYGON_REVIEW",
                "manual_verified": is_img_verified,
            })

    pd.DataFrame(checklist_rows).to_csv(LEAF_SEG_CHECKLIST_CSV, index=False)

    # [Temuan G] val_verified dan test_verified HANYA True jika 100% citra pada split tersebut
    # hadir (tidak ada kasus sulit yang dilewati) DAN seluruhnya telah disetujui manual.
    stats["val_verified"] = (
        stats["expected_val_total"] > 0
        and stats["val_manually_verified_count"] == stats["expected_val_total"]
        and stats["val_missing_or_unverified_count"] == 0
    )
    stats["test_verified"] = (
        stats["expected_test_total"] > 0
        and stats["test_manually_verified_count"] == stats["expected_test_total"]
        and stats["test_missing_or_unverified_count"] == 0
    )

    seg_manifest_path = os.path.join(LEAF_SEG_DIR, "segmentation_qa_manifest.json")
    with open(seg_manifest_path, "w", encoding="utf-8") as f:
        json.dump(stats, f, indent=2)

    with open(LEAF_SEG_YAML, "w", encoding="utf-8") as f:
        yaml.dump({
            "path": LEAF_SEG_DIR,
            "train": "images/train",
            "val": "images/val",
            "test": "images/test",
            "nc": 1,
            "names": ["tea_leaf"],
        }, f, sort_keys=False)

    print("=" * 92)
    print("  [STATUS DATASET SEGMENTASI DAUN (CELL 28 — AUDIT TEMUAN B, C, F, G)]")
    print("=" * 92)
    print(f"  - Train Polygons Layak & Terpetakan ke Split Asli : {stats['train']}/{stats['expected_train_total']} citra (Verified={stats['train_manually_verified_count']})")
    print(f"  - Val Polygons Terverifikasi Manual Lengkap       : {stats['val_manually_verified_count']}/{stats['expected_val_total']} citra (val_verified={stats['val_verified']})")
    print(f"  - Test Polygons Terverifikasi Manual Lengkap      : {stats['test_manually_verified_count']}/{stats['expected_test_total']} citra (test_verified={stats['test_verified']})")
    print(f"  - Poligon Cacat yang Ditolak                      : {stats['rejected_polygons']} poligon")
    print(f"  - Checklist Verifikasi Per Gambar                 : {LEAF_SEG_CHECKLIST_CSV}")
    return stats

SEG_DATASET_STATS = prepare_verified_leaf_segmentation_dataset()


### [Cell 30 | Prioritas 3 (Cell Baru B)] Validasi Anotasi Poligon + Kelengkapan Checklist Manual & Training `YOLO26n-seg`

In [ ]:
# ==============================================================================
# REVISI CELL 30 / CELL BARU B (Prioritas 3 | Temuan Audit C, D, G):
# 1. [Temuan C] Menambahkan `import yaml` secara eksplisit.
# 2. [Temuan G] Memeriksa validitas geometri poligon SEKALIGUS kelengkapan
#    persetujuan manual per gambar (`leaf_seg_manual_verification_checklist.csv`):
#      - Bukan format bounding box 4 angka (wajib >= 3 titik sudut / >= 6 koordinat).
#      - Seluruh daun target dalam citra telah diberi poligon (`all_target_leaves_annotated=True`).
#      - 100% citra pada split `val` dan `test` di `field_500_split_manifest.json`
#        wajib hadir dan terverifikasi (`val_verified=True` dan `test_verified=True`),
#        sehingga kasus daun sulit/bertumpuk tidak terlewat.
# 3. [Temuan D] Setelah `YOLO26n-seg` selesai dilatih, bobot khusus daun teh
#    (`LEAF_SEG_WEIGHTS`) langsung dimuat ke `field_detector` melalui
#    `field_detector.load_trained_leaf_seg(LEAF_SEG_WEIGHTS)` sebelum pengujian
#    sistem terintegrasi di Cell 18, Cell 32, dan Cell 34.
# ==============================================================================
import yaml

SEG_MODEL_YAML = os.path.join(BASE_WORK_DIR, "yolo26n_seg_leaf.yaml")
SEG_YAML_CONTENT = """nc: 1
scales:
  n: [0.50, 0.25, 1024]

backbone:
  - [-1, 1, Conv, [64, 3, 2]]
  - [-1, 1, Conv, [128, 3, 2]]
  - [-1, 2, C3k2, [256, False, 0.25]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [-1, 2, C3k2, [512, False, 0.25]]
  - [-1, 1, Conv, [512, 3, 2]]
  - [-1, 2, C3k2, [512, True]]
  - [-1, 1, Conv, [1024, 3, 2]]
  - [-1, 2, C3k2, [1024, True]]
  - [-1, 1, SPPF, [1024, 5, 3, True]]
  - [-1, 2, C2PSA, [1024]]

head:
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 6], 1, Concat, [1]]
  - [-1, 2, C3k2, [512, True]]
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 4], 1, Concat, [1]]
  - [-1, 2, C3k2, [256, True]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 13], 1, Concat, [1]]
  - [-1, 2, C3k2, [512, True]]
  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 10], 1, Concat, [1]]
  - [-1, 1, C3k2, [1024, True, 0.5, True]]
  - [[16, 19, 22], 1, Segment, [nc, 32, 256]]
"""
with open(SEG_MODEL_YAML, "w", encoding="utf-8") as f:
    f.write(SEG_YAML_CONTENT)

def validate_polygon_annotations_before_training(seg_dataset_dir: str) -> dict:
    """
    Memeriksa validitas geometri seluruh file label poligon pada split train, val, dan test,
    serta memastikan 100% citra val dan test telah disetujui manual tanpa mengabaikan kasus sulit.
    """
    report = {}
    all_splits_valid = True
    for sp in ["train", "val", "test"]:
        lbl_files = sorted(glob.glob(os.path.join(seg_dataset_dir, "labels", sp, "*.txt")))
        valid_files, valid_polys, invalid_lines = 0, 0, 0
        for lp in lbl_files:
            file_ok = True
            has_poly = False
            with open(lp, "r", encoding="utf-8") as f:
                for ln in f:
                    parts = ln.strip().split()
                    if not parts:
                        continue
                    if len(parts) < 7 or (len(parts) - 1) % 2 != 0:
                        file_ok = False
                        invalid_lines += 1
                        continue
                    cid = int(float(parts[0]))
                    coords = [float(v) for v in parts[1:]]
                    ok, _ = is_polygon_geometrically_feasible(coords)
                    if cid != 0 or not ok:
                        file_ok = False
                        invalid_lines += 1
                    else:
                        has_poly = True
                        valid_polys += 1
            if file_ok and has_poly:
                valid_files += 1
        report[sp] = {
            "total_label_files": len(lbl_files),
            "valid_label_files": valid_files,
            "valid_polygons": valid_polys,
            "invalid_lines": invalid_lines,
        }
        if valid_files == 0 or invalid_lines > 0:
            all_splits_valid = False

    qa_manifest_p = os.path.join(seg_dataset_dir, "segmentation_qa_manifest.json")
    qa_info = json.load(open(qa_manifest_p, "r", encoding="utf-8")) if os.path.exists(qa_manifest_p) else {}
    val_test_verified = bool(qa_info.get("val_verified", False) and qa_info.get("test_verified", False))
    report["val_verified"] = bool(qa_info.get("val_verified", False))
    report["test_verified"] = bool(qa_info.get("test_verified", False))
    report["ready_for_training"] = (all_splits_valid and val_test_verified)
    return report

poly_check_report = validate_polygon_annotations_before_training(LEAF_SEG_DIR)
print("=" * 92)
print("  [PEMERIKSAAN VALIDITAS POLIGON & CHECKLIST MANUAL SEBELUM TRAINING YOLO26n-seg (CELL 30)]")
print("=" * 92)
for sp in ["train", "val", "test"]:
    r_sp = poly_check_report[sp]
    print(f"  - Split {sp.upper():<5}: {r_sp['valid_label_files']}/{r_sp['total_label_files']} file valid | {r_sp['valid_polygons']} poligon daun | {r_sp['invalid_lines']} baris cacat")
print(f"  - Status Verifikasi Penuh Val/Test (100% Cakupan): val_verified={poly_check_report['val_verified']}, test_verified={poly_check_report['test_verified']}")

SEG_RUN_NAME = "yolo26n_seg_tea_leaf"
if poly_check_report["ready_for_training"]:
    seg_model_trainer = YOLO(BASE_SEG_PT if os.path.exists(BASE_SEG_PT) else SEG_MODEL_YAML)
    seg_train_results = seg_model_trainer.train(
        data=LEAF_SEG_YAML,
        epochs=35,
        imgsz=640,
        batch=16,
        device="cuda" if torch.cuda.is_available() else "cpu",
        project=PROJECT_RUNS,
        name=SEG_RUN_NAME,
        exist_ok=True,
        patience=10,
        cos_lr=True,
        retina_masks=True,
        seed=42,
        plots=True,
    )
    LEAF_SEG_WEIGHTS = os.path.join(PROJECT_RUNS, SEG_RUN_NAME, "weights", "best.pt")
    print(f"[SUKSES] Model YOLO26n-seg khusus daun teh selesai dilatih: {LEAF_SEG_WEIGHTS}")
    field_detector.load_trained_leaf_seg(LEAF_SEG_WEIGHTS)
elif os.path.exists(LEAF_SEG_WEIGHTS):
    print(f"[INFO] Memuat bobot YOLO26n-seg daun teh yang sudah ada: {LEAF_SEG_WEIGHTS}")
    field_detector.load_trained_leaf_seg(LEAF_SEG_WEIGHTS)
else:
    print("\n[TRAINING YOLO26n-seg DITANGGUHKAN]")
    print("  Dataset segmentasi belum memenuhi syarat 100% poligon terverifikasi manual pada split val/test.")
    print(f"  1. Unggah `{ROBOFLOW_SEG_ZIP}` atau simpan poligon di `{VERIFIED_SEG_FOLDER}`")
    print(f"  2. Lengkapi persetujuan manual per gambar (`annotator_id`, `all_target_leaves_annotated=True`,")
    print(f"     `review_decision='APPROVED_COMPLETE_POLYGONS'`) di `{LEAF_SEG_CHECKLIST_CSV}`")
    print("  3. Jalankan ulang Cell 28 dan Cell 30 sebelum melanjutkan ke Cell 18, Cell 32, dan Cell 34.")


### [Cell 32 | Prioritas 2 (Cell Baru C)] Leaf–Lesion Association & Penanganan Eksplisit Bercak di Antara Dua Daun (`AMBIGUOUS_OVERLAP`)

In [ ]:
# ==============================================================================
# REVISI CELL 32 / CELL BARU C (Prioritas 2 | Temuan Audit D):
# 1. Memastikan model segmentasi khusus daun teh (`LEAF_SEG_WEIGHTS` dari Cell 30)
#    telah dimuat sebelum menjalankan Leaf-Lesion Association.
# 2. Menghitung matriks Lesion Containment Ratio (LCR) berukuran
#    (N_lesions x N_leaves): LCR(B_j, M_i) = |B_j ∩ M_i| / |B_j|.
# 3. Menangani secara eksplisit bercak yang berada di antara dua daun atau
#    memiliki overlap ambigu (`AMBIGUOUS_OVERLAP`) agar tidak dipaksakan ke
#    daun yang salah.
# ==============================================================================
def compute_containment_matrix_and_audit(img_bgr: np.ndarray, detector: TeaLeafFieldDetector):
    leaves = detector.detect_leaves(img_bgr, conf=0.25, allow_hsv_fallback=False)
    lesions = detector.detect_lesions(img_bgr, conf=0.25, use_tta=True, exclude_healthy_leaf_class=True)
    assoc = detector.associate_leaves_and_lesions(leaves, lesions, min_containment=0.30, ambiguity_margin=0.20)

    n_les = len(lesions)
    n_lf = len(leaves)
    c_mat = np.zeros((n_les, n_lf), dtype=np.float32)

    for j, les in enumerate(lesions):
        bx1, by1, bx2, by2 = les["bbox"]
        b_area = max(1, (bx2 - bx1) * (by2 - by1))
        for i, lf in enumerate(leaves):
            m_crop = lf["mask"][max(0, by1):max(0, by2), max(0, bx1):max(0, bx2)]
            inter = int(m_crop.sum()) if m_crop.size > 0 else 0
            c_mat[j, i] = inter / float(b_area)

    return c_mat, assoc

if field_detector.leaf_seg_model is None and os.path.exists(LEAF_SEG_WEIGHTS):
    field_detector.load_trained_leaf_seg(LEAF_SEG_WEIGHTS)

sample_field_imgs = sorted(glob.glob(os.path.join(FIELD_500_DIR, "images", "test", "*.*")))
if field_detector.leaf_seg_model is None:
    print("[PERINGATAN URUTAN EKSEKUSI (Temuan D)] Model `YOLO26n-seg` khusus daun teh belum dilatih.")
    print("  Jalankan Cell 28 dan Cell 30 terlebih dahulu sebelum menjalankan audit Leaf-Lesion Association di Cell 32.")
elif sample_field_imgs:
    img_demo = cv2.imread(sample_field_imgs[0])
    c_mat, assoc_demo = compute_containment_matrix_and_audit(img_demo, field_detector)
    vis_demo = visualize_field_analysis(img_demo, assoc_demo)

    fig, axes = plt.subplots(1, 2, figsize=(16, 6.5))
    axes[0].imshow(cv2.cvtColor(vis_demo, cv2.COLOR_BGR2RGB))
    axes[0].axis("off")
    axes[0].set_title(
        f"Hasil Leaf-Lesion Association\n"
        f"Daun: {len(assoc_demo['leaves'])} | Bercak: {len(assoc_demo['lesions'])} | Ambigu: {len(assoc_demo['ambiguous_lesions'])}",
        fontweight="bold",
    )

    if c_mat.size > 0:
        im = axes[1].imshow(c_mat, cmap="YlGnBu", vmin=0.0, vmax=1.0, aspect="auto")
        axes[1].set_xticks(range(len(assoc_demo["leaves"])))
        axes[1].set_xticklabels([f"Leaf #{lf['leaf_id']}" for lf in assoc_demo["leaves"]])
        axes[1].set_yticks(range(len(assoc_demo["lesions"])))
        axes[1].set_yticklabels([f"Lesion #{l['lesion_id']} ({l['cls_name'][:8]})" for l in assoc_demo["lesions"]])
        for r_i in range(c_mat.shape[0]):
            for c_i in range(c_mat.shape[1]):
                axes[1].text(c_i, r_i, f"{c_mat[r_i, c_i]:.2f}", ha="center", va="center", color="black", fontsize=9)
        plt.colorbar(im, ax=axes[1], label="Lesion Containment Ratio (LCR)")
        axes[1].set_title("Matriks LCR(B_j, M_i) & Deteksi Overlap Ambigu", fontweight="bold")
    else:
        axes[1].text(0.5, 0.5, "Tidak ada pasangan daun-bercak pada sampel ini", ha="center", va="center")
        axes[1].axis("off")

    plt.tight_layout()
    plt.show()

    assoc_audit_rows = []
    for lf in assoc_demo["leaves"]:
        for les in lf["lesions"]:
            assoc_audit_rows.append({
                "Lesion ID": f"Lesion #{les['lesion_id']}",
                "Kelas Penyakit": les["cls_name"],
                "Status Asosiasi": les["association_status"],
                "Assigned Leaf": f"Leaf #{les['leaf_id']}",
                "Kandidat Daun": str(les["candidate_leaf_ids"]),
                "LCR Top-1": les["containment_ratio"],
                "LCR Top-2": les["second_containment_ratio"],
                "Selisih LCR": les["lcr_margin"],
            })
    for amb in assoc_demo.get("ambiguous_lesions", []):
        assoc_audit_rows.append({
            "Lesion ID": f"Lesion #{amb['lesion_id']}",
            "Kelas Penyakit": amb["cls_name"],
            "Status Asosiasi": "AMBIGUOUS_OVERLAP (Tidak Dipaksakan)",
            "Assigned Leaf": "None (Ambiguous)",
            "Kandidat Daun": str(amb["candidate_leaf_ids"]),
            "LCR Top-1": amb["containment_ratio"],
            "LCR Top-2": amb["second_containment_ratio"],
            "Selisih LCR": amb["lcr_margin"],
        })
    if assoc_audit_rows:
        display(pd.DataFrame(assoc_audit_rows))


### [Cell 34 | Prioritas 1 — WAJIB (Cell Baru D)] Evaluasi Sistem Terintegrasi Berdasarkan Ground Truth Pasangan Daun–Bercak yang Terverifikasi Manual (`gt_leaf_lesion_pairs_test_verified.csv`)

In [ ]:
# ==============================================================================
# REVISI CELL 34 / CELL BARU D (Prioritas 1 — WAJIB | Temuan Audit A, D, H):
# 1. [Temuan A] Memperbaiki filter kelas penyakit Ground Truth pada evaluasi
#    pasangan daun-bercak:
#    - TIDAK mengecualikan kelas 3 (`Helopeltis`).
#    - Menggunakan `g['cls'] in DISEASE_LESION_CLASS_IDS` (kelas 0..5, termasuk
#      kelas 3 `Helopeltis`) dan menyaring kelas 6 (`HEALTHY_LEAF_CLASS_ID`).
# 2. [Temuan H] Ground Truth pasangan daun-bercak (`GT_Leaf`, `GT_Lesion`) pada
#    test set WAJIB diverifikasi manual melalui manifest
#    `gt_leaf_lesion_pairs_test_verified.csv`, terutama pada kasus daun yang
#    saling bertumpuk (`is_overlapping_leaf_case=True`):
#    - Sistem mengekspor kandidat pasangan beserta nilai `top1_lcr`, `top2_lcr`,
#      dan penanda `is_overlapping_leaf_case` untuk diperiksa manusia.
#    - Evaluasi akurasi asosiasi HANYA menggunakan pasangan GT yang telah
#      disetujui manual (`manual_verified=True`, `annotator_id != ''`,
#      `review_decision='APPROVED_GT_PAIR'`, dan `verified_gt_leaf_id` valid),
#      bukan sekadar mengambil `best_lcr >= 0.30` secara otomatis.
# ==============================================================================
from collections import defaultdict

GT_PAIR_MANIFEST_CSV  = os.path.join(FIELD_500_DIR, "gt_leaf_lesion_pairs_test_verified.csv")
EXTERNAL_GT_PAIR_CSV  = "/content/gt_leaf_lesion_pairs_test_verified.csv"

def load_gt_leaf_masks_from_polygons(seg_lbl_path: str, h: int, w: int) -> list:
    gt_leaves = []
    if not os.path.exists(seg_lbl_path):
        return gt_leaves
    with open(seg_lbl_path, "r", encoding="utf-8") as f:
        for idx_k, ln in enumerate(f, 1):
            parts = ln.strip().split()
            if len(parts) < 7 or (len(parts) - 1) % 2 != 0:
                continue
            coords = np.array([float(v) for v in parts[1:]], dtype=np.float32).reshape(-1, 2)
            pts_px = np.column_stack([coords[:, 0] * w, coords[:, 1] * h]).astype(np.int32)
            m = np.zeros((h, w), dtype=np.uint8)
            cv2.fillPoly(m, [pts_px], 1)
            if int(m.sum()) > 0:
                gt_leaves.append({"gt_leaf_id": idx_k, "mask": m})
    return gt_leaves

def compute_binary_mask_iou(mask_a: np.ndarray, mask_b: np.ndarray) -> float:
    inter = int(np.logical_and(mask_a > 0, mask_b > 0).sum())
    union = int(np.logical_or(mask_a > 0, mask_b > 0).sum())
    return inter / float(max(1, union))

def build_or_load_verified_gt_leaf_lesion_pairs(test_imgs: list) -> tuple:
    """
    Membangun atau memuat manifest verifikasi manual pasangan (GT_Leaf, GT_Lesion)
    pada split test (`gt_leaf_lesion_pairs_test_verified.csv`).
    Menandai secara khusus kasus daun yang saling bertumpuk (`is_overlapping_leaf_case`)
    dan mewajibkan verifikasi manual (`manual_verified=True`, `annotator_id != ''`)
    sebelum pasangan GT digunakan dalam perhitungan akurasi asosiasi.
    """
    existing_pair_signoff = {}
    for cand_csv in [EXTERNAL_GT_PAIR_CSV, GT_PAIR_MANIFEST_CSV]:
        if os.path.exists(cand_csv):
            try:
                df_prev = pd.read_csv(cand_csv).fillna("")
                for _, r in df_prev.iterrows():
                    p_key = str(r.get("gt_pair_key", "")).strip()
                    if p_key:
                        existing_pair_signoff[p_key] = r.to_dict()
            except Exception:
                pass

    manifest_rows = []
    verified_gt_pairs_by_stem = defaultdict(list)
    stats_pairs = {
        "total_candidate_gt_lesions": 0,
        "overlapping_leaf_cases": 0,
        "manually_verified_gt_pairs": 0,
        "unverified_gt_pairs": 0,
        "all_gt_pairs_verified": False,
    }

    for img_p in test_imgs:
        bname = os.path.basename(img_p)
        stem = os.path.splitext(bname)[0]
        gt_lesion_lbl = os.path.join(FIELD_500_DIR, "labels", "test", stem + ".txt")
        gt_leaf_lbl   = os.path.join(LEAF_SEG_DIR, "labels", "test", stem + ".txt")
        if not (os.path.exists(gt_lesion_lbl) and os.path.exists(gt_leaf_lbl)):
            continue

        img_bgr = cv2.imread(img_p)
        if img_bgr is None:
            continue
        h, w = img_bgr.shape[:2]
        gt_leaves = load_gt_leaf_masks_from_polygons(gt_leaf_lbl, h, w)
        if not gt_leaves:
            continue
        gt_leaf_map = {lf["gt_leaf_id"]: lf["mask"] for lf in gt_leaves}

        # [Temuan A] Gunakan DISEASE_LESION_CLASS_IDS (0..5, termasuk 3 = Helopeltis; kecualikan 6 = Healty Leaf)
        gt_lesions = [g for g in load_all_gt_boxes(gt_lesion_lbl, w, h) if g["cls"] in DISEASE_LESION_CLASS_IDS]

        for g_idx, g_les in enumerate(gt_lesions):
            stats_pairs["total_candidate_gt_lesions"] += 1
            gx1, gy1, gx2, gy2 = g_les["xyxy"]
            g_area = max(1, (gx2 - gx1) * (gy2 - gy1))
            cand_overlaps = []
            for g_lf in gt_leaves:
                inter = int(g_lf["mask"][max(0, gy1):max(0, gy2), max(0, gx1):max(0, gx2)].sum())
                lcr = inter / float(g_area)
                if lcr > 0.0:
                    cand_overlaps.append((lcr, g_lf["gt_leaf_id"]))
            cand_overlaps.sort(key=lambda x: x[0], reverse=True)

            top1_lcr = cand_overlaps[0][0] if len(cand_overlaps) >= 1 else 0.0
            top1_id  = cand_overlaps[0][1] if len(cand_overlaps) >= 1 else -1
            top2_lcr = cand_overlaps[1][0] if len(cand_overlaps) >= 2 else 0.0
            top2_id  = cand_overlaps[1][1] if len(cand_overlaps) >= 2 else -1

            # Deteksi apakah bercak berada pada area daun yang saling bertumpuk / overlap
            is_overlap_case = (top2_lcr >= 0.10) or (len(cand_overlaps) >= 2 and (top1_lcr - top2_lcr) < 0.25)
            if is_overlap_case:
                stats_pairs["overlapping_leaf_cases"] += 1

            pair_key = f"{stem}__lesion_{g_idx}"
            prev_r = existing_pair_signoff.get(pair_key, {})
            annotator_id = str(prev_r.get("annotator_id", "")).strip()
            review_decision = str(prev_r.get("review_decision", "PENDING_MANUAL_PAIR_VERIFICATION")).strip()
            raw_verified_id = str(prev_r.get("verified_gt_leaf_id", "")).strip()
            try:
                verified_leaf_id = int(float(raw_verified_id)) if raw_verified_id != "" else -1
            except ValueError:
                verified_leaf_id = -1

            is_pair_verified = (
                bool(annotator_id)
                and review_decision == "APPROVED_GT_PAIR"
                and verified_leaf_id in gt_leaf_map
            )

            if is_pair_verified:
                stats_pairs["manually_verified_gt_pairs"] += 1
                verified_gt_pairs_by_stem[stem].append({
                    "gt_pair_id": g_idx,
                    "gt_pair_key": pair_key,
                    "gt_cls": g_les["cls"],
                    "gt_lesion_xyxy": g_les["xyxy"],
                    "gt_leaf_id": verified_leaf_id,
                    "gt_leaf_mask": gt_leaf_map[verified_leaf_id],
                    "is_overlapping_leaf_case": is_overlap_case,
                })
            else:
                stats_pairs["unverified_gt_pairs"] += 1

            manifest_rows.append({
                "stem": stem,
                "filename": bname,
                "gt_pair_key": pair_key,
                "gt_lesion_idx": g_idx,
                "gt_cls_id": g_les["cls"],
                "gt_cls_name": CLASS_NAMES[g_les["cls"]] if 0 <= g_les["cls"] < len(CLASS_NAMES) else str(g_les["cls"]),
                "gt_lesion_xyxy": str(list(g_les["xyxy"])),
                "suggested_gt_leaf_id_for_review": top1_id,
                "second_candidate_gt_leaf_id": top2_id,
                "top1_lcr": round(top1_lcr, 4),
                "top2_lcr": round(top2_lcr, 4),
                "is_overlapping_leaf_case": is_overlap_case,
                "verified_gt_leaf_id": verified_leaf_id if is_pair_verified else "",
                "annotator_id": annotator_id,
                "review_decision": review_decision if is_pair_verified else "PENDING_MANUAL_PAIR_VERIFICATION",
                "manual_verified": is_pair_verified,
            })

    if manifest_rows:
        pd.DataFrame(manifest_rows).to_csv(GT_PAIR_MANIFEST_CSV, index=False)

    stats_pairs["all_gt_pairs_verified"] = (
        stats_pairs["total_candidate_gt_lesions"] > 0
        and stats_pairs["manually_verified_gt_pairs"] == stats_pairs["total_candidate_gt_lesions"]
        and stats_pairs["unverified_gt_pairs"] == 0
    )
    return verified_gt_pairs_by_stem, stats_pairs

def evaluate_integrated_system_against_gt_pairs(detector: TeaLeafFieldDetector) -> pd.DataFrame:
    rows = []

    # 1. Metrik Segmentasi Daun (YOLO26n-seg pada Test Set Poligon Terverifikasi Penuh)
    qa_manifest_p = os.path.join(LEAF_SEG_DIR, "segmentation_qa_manifest.json")
    qa_info = json.load(open(qa_manifest_p, "r", encoding="utf-8")) if os.path.exists(qa_manifest_p) else {}
    test_seg_verified = bool(qa_info.get("test_verified", False))
    seg_test_lbls = glob.glob(os.path.join(LEAF_SEG_DIR, "labels", "test", "*.txt"))

    if os.path.exists(LEAF_SEG_WEIGHTS) and test_seg_verified and len(seg_test_lbls) > 0:
        seg_val = YOLO(LEAF_SEG_WEIGHTS).val(data=LEAF_SEG_YAML, split="test", imgsz=640, verbose=False)
        seg_mp = float(seg_val.seg.mp) if hasattr(seg_val, "seg") else float(seg_val.box.mp)
        seg_mr = float(seg_val.seg.mr) if hasattr(seg_val, "seg") else float(seg_val.box.mr)
        seg_m50 = float(seg_val.seg.map50) if hasattr(seg_val, "seg") else float(seg_val.box.map50)
        seg_m95 = float(seg_val.seg.map) if hasattr(seg_val, "seg") else float(seg_val.box.map)
        rows.append({
            "Komponen Sistem": "1. Segmentasi Tiap Daun (YOLO26n-seg vs GT Poligon Daun 100% Verified)",
            "Precision": f"{seg_mp:.4f}",
            "Recall": f"{seg_mr:.4f}",
            "F1 / Rate": f"{2*seg_mp*seg_mr/(seg_mp+seg_mr+1e-9):.4f}",
            "mAP@0.50": f"{seg_m50:.4f}",
            "mAP@0.50:0.95": f"{seg_m95:.4f}",
            "Detail Audit": f"{len(seg_test_lbls)} citra test (100% terverifikasi manual)",
        })
    else:
        rows.append({
            "Komponen Sistem": "1. Segmentasi Tiap Daun (YOLO26n-seg vs GT Poligon Daun 100% Verified)",
            "Precision": "Menunggu GT Poligon Test Terverifikasi Penuh",
            "Recall": "-",
            "F1 / Rate": "-",
            "mAP@0.50": "-",
            "mAP@0.50:0.95": "-",
            "Detail Audit": "Pastikan test_verified=True di Cell 28 & latih model di Cell 30",
        })

    # 2. Metrik Deteksi Bercak Penyakit (YOLO26n + CA)
    rows.append({
        "Komponen Sistem": "2. Deteksi Bercak Penyakit (Single-Leaf Test GT Lengkap)",
        "Precision": f"{BASELINE_SINGLE_LEAF_METRICS['precision']:.4f}",
        "Recall": f"{BASELINE_SINGLE_LEAF_METRICS['recall']:.4f}",
        "F1 / Rate": f"{BASELINE_SINGLE_LEAF_METRICS['f1']:.4f}",
        "mAP@0.50": f"{BASELINE_SINGLE_LEAF_METRICS['map50']:.4f}",
        "mAP@0.50:0.95": f"{BASELINE_SINGLE_LEAF_METRICS['map50_95']:.4f}",
        "Detail Audit": "100% Ground Truth lengkap (termasuk Helopeltis kelas 3)",
    })

    # 3. Evaluasi Pairwise Leaf-Lesion Association terhadap Ground Truth Pasangan Daun-Bercak Terverifikasi Manual
    test_imgs = sorted(glob.glob(os.path.join(FIELD_500_DIR, "images", "test", "*.*")))
    verified_gt_pairs_by_stem, stats_pairs = build_or_load_verified_gt_leaf_lesion_pairs(test_imgs)

    if detector.leaf_seg_model is not None and stats_pairs["all_gt_pairs_verified"]:
        total_gt_pairs = 0
        total_gt_overlap_pairs = 0
        total_pred_associated_pairs = 0
        tp_pairs = 0
        tp_overlap_pairs = 0
        wrong_leaf_misassociations = 0
        ambiguous_overlap_count = 0
        evaluated_pair_images = 0

        for img_p in test_imgs:
            bname = os.path.basename(img_p)
            stem = os.path.splitext(bname)[0]
            if stem not in verified_gt_pairs_by_stem:
                continue
            img_bgr = cv2.imread(img_p)
            if img_bgr is None:
                continue

            gt_pairs = verified_gt_pairs_by_stem[stem]
            evaluated_pair_images += 1
            total_gt_pairs += len(gt_pairs)
            total_gt_overlap_pairs += sum(1 for gp in gt_pairs if gp["is_overlapping_leaf_case"])

            res_sys = detector.analyze_field_image(img_bgr, leaf_conf=0.25, lesion_conf=0.25, use_tta=True)
            ambiguous_overlap_count += len(res_sys.get("ambiguous_lesions", []))

            pred_pairs = []
            for p_lf in res_sys["leaves"]:
                for p_les in p_lf["lesions"]:
                    pred_pairs.append({
                        "pred_cls": p_les["cls_id"],
                        "pred_conf": p_les["conf"],
                        "pred_lesion_xyxy": p_les["bbox"],
                        "pred_leaf_mask": p_lf["mask"],
                        "pred_leaf_id": p_lf["leaf_id"],
                    })
            pred_pairs.sort(key=lambda x: x["pred_conf"], reverse=True)
            total_pred_associated_pairs += len(pred_pairs)

            matched_gt_pair_ids = set()
            for pp in pred_pairs:
                best_box_iou, best_gp = 0.0, None
                for gp in gt_pairs:
                    if gp["gt_pair_id"] in matched_gt_pair_ids or gp["gt_cls"] != pp["pred_cls"]:
                        continue
                    biou = compute_box_iou_xyxy(pp["pred_lesion_xyxy"], gp["gt_lesion_xyxy"])
                    if biou > best_box_iou:
                        best_box_iou, best_gp = biou, gp

                if best_gp is not None and best_box_iou >= 0.50:
                    leaf_mask_iou = compute_binary_mask_iou(pp["pred_leaf_mask"], best_gp["gt_leaf_mask"])
                    if leaf_mask_iou >= 0.50:
                        matched_gt_pair_ids.add(best_gp["gt_pair_id"])
                        tp_pairs += 1
                        if best_gp["is_overlapping_leaf_case"]:
                            tp_overlap_pairs += 1
                    else:
                        wrong_leaf_misassociations += 1

        pair_prec = tp_pairs / float(max(1, total_pred_associated_pairs))
        pair_rec  = tp_pairs / float(max(1, total_gt_pairs))
        pair_f1   = 2 * pair_prec * pair_rec / (pair_prec + pair_rec + 1e-9)
        rows.append({
            "Komponen Sistem": "3. Pairwise Leaf-Lesion Association (vs GT Pasangan Terverifikasi Manual)",
            "Precision": f"{pair_prec:.4f}",
            "Recall": f"{pair_rec:.4f}",
            "F1 / Rate": f"{pair_f1:.4f}",
            "mAP@0.50": f"TP_Pair={tp_pairs}/{total_gt_pairs}",
            "mAP@0.50:0.95": f"WrongLeaf={wrong_leaf_misassociations}",
            "Detail Audit": f"OverlapCases={tp_overlap_pairs}/{total_gt_overlap_pairs} | Ambiguous={ambiguous_overlap_count}",
        })
    else:
        rows.append({
            "Komponen Sistem": "3. Pairwise Leaf-Lesion Association (vs GT Pasangan Terverifikasi Manual)",
            "Precision": "Menunggu Verifikasi Manual Pasangan GT Daun-Bercak",
            "Recall": "-",
            "F1 / Rate": "-",
            "mAP@0.50": f"Verified={stats_pairs['manually_verified_gt_pairs']}/{stats_pairs['total_candidate_gt_lesions']}",
            "mAP@0.50:0.95": f"OverlapCases={stats_pairs['overlapping_leaf_cases']}",
            "Detail Audit": f"Verifikasi `verified_gt_leaf_id` & `annotator_id` di {GT_PAIR_MANIFEST_CSV}",
        })

    return pd.DataFrame(rows)

df_integrated_eval = evaluate_integrated_system_against_gt_pairs(field_detector)
print("\n" + "=" * 105)
print("  [TABEL 34] EVALUASI SISTEM TERINTEGRASI BERDASARKAN GROUND TRUTH PASANGAN DAUN-BERCAK TERVERIFIKASI MANUAL")
print("=" * 105)
display(df_integrated_eval)
